# Phase 4 Complete Final — Global Fusion and Confidence-Gated Hybrid

This notebook contains Part A: A global reciprocal-rank fusion and cluster-ablation experiment, Part B: A primary confidence-gated experiment, and Part C: final results and artifact consolidation.

The selected architecture is CF with bounded content reranking, sequential routing when CF is unavailable, and discovery-early popularity as final fallback.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1 — Imports, isolated paths and configuration


In [4]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BENCHMARK_ROOT = Path('/content/drive/MyDrive/datasets/recommendation_benchmark_final_outputs')
BASELINE_ROOT = Path('/content/drive/MyDrive/datasets/recommendation_baseline_outputs')
OUTPUT_ROOT = Path('/content/drive/MyDrive/datasets/hybrid_recommender_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATASET_DIRS = {
    'ASSISTments': BENCHMARK_ROOT / 'assistments',
    'KDD': BENCHMARK_ROOT / 'kdd',
}
RANDOM_STATE = 42
DISCOVERY_TUNING_FRACTION = 0.20
METRIC_KS = (5, 10, 20)
MAX_RECOMMENDATIONS = max(METRIC_KS)
PRIMARY_DATASET = 'ASSISTments'
PRIMARY_TASK = 'problem'
PRIMARY_RELEVANCE = 'attempted'
PRIMARY_CANDIDATE_POLICY = 'all_supported'
PRIMARY_SEGMENT = 'all'
PRIMARY_K = 10
BOOTSTRAP_RESAMPLES = 2000
COVERAGE_FLOOR_RATIO = 0.50
EXPECTED_PHASE3_RECOMMENDATION_ROWS = 6_125_933
RRF_CONSTANT = 60

assert BENCHMARK_ROOT != BASELINE_ROOT != OUTPUT_ROOT
assert OUTPUT_ROOT not in {BENCHMARK_ROOT, BASELINE_ROOT}
print(f'Phase 2 input: {BENCHMARK_ROOT}')
print(f'Phase 3 input: {BASELINE_ROOT}')
print(f'Phase 4 output: {OUTPUT_ROOT}')

Phase 2 input: /content/drive/MyDrive/datasets/recommendation_benchmark_final_outputs
Phase 3 input: /content/drive/MyDrive/datasets/recommendation_baseline_outputs
Phase 4 output: /content/drive/MyDrive/datasets/hybrid_recommender_outputs


## Step 2 — Validate the Phase 2 and verified Phase 3 handoff

In [5]:
PHASE2_PARQUET_FILES = {
    'learner_splits.parquet', 'skill_catalog.parquet',
    'problem_catalog.parquet', 'early_skill_history.parquet',
    'future_skill_relevance.parquet', 'early_problem_history.parquet',
    'future_problem_relevance.parquet', 'problem_skill_map.parquet',
    'skill_name_id_map.parquet',
}
PHASE3_ARTIFACTS = {
    'recommendation_metrics.csv', 'recommendation_coverage.csv',
    'model_comparison.csv', 'validation_recommendations.parquet',
    'popularity_scores.parquet', 'content_scores.parquet',
    'neighbor_configuration.csv', 'transition_probabilities.parquet',
    'hyperparameter_tuning_metrics.csv',
    'hyperparameter_selection.csv', 'leakage_audit.csv',
    'phase3_config.json',
}
PHASE3_RECOMMENDATION_COLUMNS = {
    'Dataset', 'Task', 'Model', 'CandidatePolicy',
    'RelevanceDefinition', 'learner_id', 'item_id', 'score', 'rank',
}

def artifact_row_count(path):
    if path.suffix == '.parquet':
        return pq.ParquetFile(path).metadata.num_rows
    if path.suffix == '.csv':
        return len(pd.read_csv(path))
    if path.suffix == '.json':
        return 1
    raise ValueError(f'Unsupported artifact type: {path}')

for filename in [
    'benchmark_summary.csv', 'artifact_manifest.csv',
    'benchmark_config.json', 'benchmark_schema.json',
]:
    assert (BENCHMARK_ROOT / filename).is_file(), filename
assert (BASELINE_ROOT / 'artifact_manifest.csv').is_file()

with open(BENCHMARK_ROOT / 'benchmark_config.json', encoding='utf-8') as file:
    benchmark_config = json.load(file)
benchmark_summary = pd.read_csv(BENCHMARK_ROOT / 'benchmark_summary.csv')
phase2_manifest = pd.read_csv(BENCHMARK_ROOT / 'artifact_manifest.csv')
assert benchmark_config['candidate_statistics_source'] == 'discovery_early_only'
assert benchmark_config['future_role'] == 'relevance_labels_only'
assert benchmark_config['assist_skill_name_mapping_source'] == 'discovery_early_only'
assert len(phase2_manifest) == 18
assert set(phase2_manifest['Dataset']) == set(DATASET_DIRS)

phase2_contract_rows = []
for dataset, directory in DATASET_DIRS.items():
    dataset_manifest = phase2_manifest[
        phase2_manifest['Dataset'].eq(dataset)
    ].set_index('File')
    assert set(dataset_manifest.index) == PHASE2_PARQUET_FILES
    for filename in sorted(PHASE2_PARQUET_FILES):
        path = directory / filename
        assert path.is_file(), path
        metadata = pq.ParquetFile(path).metadata
        manifest_row = dataset_manifest.loc[filename]
        assert metadata.num_rows == int(manifest_row['Rows'])
        assert path.stat().st_size == int(manifest_row['Bytes'])
        phase2_contract_rows.append({
            'Dataset': dataset, 'File': filename,
            'Rows': metadata.num_rows, 'Bytes': path.stat().st_size,
            'ManifestValid': True,
        })
phase2_contract = pd.DataFrame(phase2_contract_rows)

phase3_manifest = pd.read_csv(BASELINE_ROOT / 'artifact_manifest.csv')
assert len(phase3_manifest) == len(PHASE3_ARTIFACTS)
assert set(phase3_manifest['File']) == PHASE3_ARTIFACTS
phase3_manifest_lookup = phase3_manifest.set_index('File')
phase3_contract_rows = []
for filename in sorted(PHASE3_ARTIFACTS):
    path = BASELINE_ROOT / filename
    assert path.is_file(), path
    manifest_row = phase3_manifest_lookup.loc[filename]
    rows = artifact_row_count(path)
    assert rows == int(manifest_row['Rows'])
    assert path.stat().st_size == int(manifest_row['Bytes'])
    phase3_contract_rows.append({
        'File': filename, 'Rows': rows, 'Bytes': path.stat().st_size,
        'ManifestValid': True,
    })
phase3_contract = pd.DataFrame(phase3_contract_rows)
recommendation_schema = set(
    pq.ParquetFile(BASELINE_ROOT / 'validation_recommendations.parquet')
    .schema_arrow.names
)
assert PHASE3_RECOMMENDATION_COLUMNS.issubset(recommendation_schema)

with open(BASELINE_ROOT / 'phase3_config.json', encoding='utf-8') as file:
    phase3_config = json.load(file)
assert phase3_config['phase'] == 3
assert phase3_config['implemented_steps'] == list(range(1, 17))
assert phase3_config['selection_source'] == 'discovery_only'
assert phase3_config['final_evaluation_cohort'] == 'validation'
assert phase3_config['validation_future_role'] == 'evaluation_only'
assert phase3_config['dense_learner_item_matrix_constructed'] is False

leakage_audit = pd.read_csv(BASELINE_ROOT / 'leakage_audit.csv')
LEAKAGE_FLAGS = [
    'DiscoveryValidationDisjoint', 'RecommendationsValidationOnly',
    'CandidatesInFrozenCatalog', 'NovelRecommendationsExcludeSeen',
    'CatalogSupportsMaximumK',
]
assert len(leakage_audit) == 4
assert set(LEAKAGE_FLAGS).issubset(leakage_audit.columns)
assert leakage_audit[LEAKAGE_FLAGS].apply(
    lambda column: column.astype(str).str.lower().eq('true')
).all().all()
phase3_recommendation_rows = pq.ParquetFile(
    BASELINE_ROOT / 'validation_recommendations.parquet'
).metadata.num_rows
assert phase3_recommendation_rows == EXPECTED_PHASE3_RECOMMENDATION_ROWS
assert leakage_audit['RecommendationRowsChecked'].sum() == phase3_recommendation_rows

neighbor_configuration = pd.read_csv(
    BASELINE_ROOT / 'neighbor_configuration.csv'
)
assert len(neighbor_configuration) == 4
assert neighbor_configuration['SelectedNeighbors'].eq(20).all()
assert neighbor_configuration['SelectedNeighborWeighting'].eq('cosine').all()
hyperparameter_selection = pd.read_csv(
    BASELINE_ROOT / 'hyperparameter_selection.csv'
)
expected_sequence = {
    ('ASSISTments', 'problem'): (0.0, 0.5),
    ('ASSISTments', 'skill'): (10.0, 0.5),
    ('KDD', 'problem'): (0.0, 0.5),
    ('KDD', 'skill'): (0.0, 0.9),
}
for row in hyperparameter_selection.itertuples(index=False):
    expected_shrinkage, expected_decay = expected_sequence[(row.Dataset, row.Task)]
    assert np.isclose(row.SelectedTransitionShrinkage, expected_shrinkage)
    assert np.isclose(row.SelectedRecencyDecay, expected_decay)
    if row.Task == 'problem':
        assert np.isclose(row.SelectedDifficultyTolerance, 1.0)

model_comparison = pd.read_csv(BASELINE_ROOT / 'model_comparison.csv')
assert len(model_comparison) == 28
assist_decisions = model_comparison[model_comparison['Dataset'].eq('ASSISTments')]
decision_lookup = assist_decisions.set_index(['Task', 'Model'])['Decision'].to_dict()
for task in ['problem', 'skill']:
    assert decision_lookup[(task, 'learner_neighbor_cf')] == 'eligible_for_phase4'
    assert decision_lookup[(task, 'sequential_transition')] == 'eligible_for_phase4'
    assert decision_lookup[(task, 'same_cluster_neighbor_cf')] == 'exclude_cluster_signal'
    assert decision_lookup[(task, 'cluster_popularity')] == 'exclude_cluster_signal'
assert decision_lookup[('problem', 'content_problem')] == 'does_not_pass_phase3_gate'
assert decision_lookup[('skill', 'weak_skill_mastery_confidence')] == 'does_not_pass_phase3_gate'
kdd_decisions = model_comparison[model_comparison['Dataset'].eq('KDD')].copy()
assert kdd_decisions.loc[
    kdd_decisions['Model'].eq('popularity_discovery_early'), 'Decision'
].eq('reference_baseline').all()
assert kdd_decisions.loc[
    ~kdd_decisions['Model'].eq('popularity_discovery_early'), 'Decision'
].eq('secondary_evidence_only').all()

display(phase2_contract)
display(phase3_contract)
display(model_comparison.sort_values(['Dataset', 'Task', 'NDCGAtK'], ascending=[True, True, False]))

,Dataset,File,Rows,Bytes,ManifestValid
0,ASSISTments,early_problem_history.parquet,4052498,76911901,True
1,ASSISTments,early_skill_history.parquet,260768,8707997,True
2,ASSISTments,future_problem_relevance.parquet,1783782,35311927,True
3,ASSISTments,future_skill_relevance.parquet,174823,4960293,True
4,ASSISTments,learner_splits.parquet,33335,1283461,True
5,ASSISTments,problem_catalog.parquet,39779,978772,True
6,ASSISTments,problem_skill_map.parquet,17826,176578,True
7,ASSISTments,skill_catalog.parquet,161,16637,True
8,ASSISTments,skill_name_id_map.parquet,194,10524,True
9,KDD,early_problem_history.parquet,46670,1410449,True


,File,Rows,Bytes,ManifestValid
0,content_scores.parquet,6770282,145442689,True
1,hyperparameter_selection.csv,4,557,True
2,hyperparameter_tuning_metrics.csv,125,36764,True
3,leakage_audit.csv,4,367,True
4,model_comparison.csv,28,14468,True
5,neighbor_configuration.csv,4,500,True
6,phase3_config.json,1,928,True
7,popularity_scores.parquet,163576,1193533,True
8,recommendation_coverage.csv,840,103347,True
9,recommendation_metrics.csv,840,235222,True


,Dataset,Task,CandidatePolicy,RelevanceDefinition,Model,Segment,K,PrecisionAtK,RecallAtK,NDCGAtK,...,RecallDeltaVsPopularity,NDCGDeltaVsPopularity,CoverageRatioVsPopularity,BeatsPopularity,OrdinaryCFRecallAtK,OrdinaryCFNDCGAtK,OrdinaryCFCatalogCoverageAtK,BeatsOrdinaryCF,Decision,DecisionScope
20,ASSISTments,problem,all_supported,attempted,learner_neighbor_cf,all,10,0.439626,0.270136,0.498392,...,0.262903,0.486491,873.2,True,0.270136,0.498392,0.219513,False,eligible_for_phase4,primary
10,ASSISTments,problem,all_supported,attempted,same_cluster_neighbor_cf,all,10,0.430871,0.263530,0.488105,...,0.256296,0.476205,904.0,True,0.270136,0.498392,0.219513,False,exclude_cluster_signal,primary
24,ASSISTments,problem,all_supported,attempted,sequential_transition,all,10,0.157305,0.116604,0.219903,...,0.109370,0.208003,1515.6,True,0.270136,0.498392,0.219513,False,eligible_for_phase4,primary
11,ASSISTments,problem,all_supported,attempted,cluster_popularity,all,10,0.031004,0.017859,0.031371,...,0.010625,0.019470,2.0,True,0.270136,0.498392,0.219513,False,exclude_cluster_signal,primary
1,ASSISTments,problem,all_supported,attempted,popularity_discovery_future,all,10,0.031488,0.003712,0.031343,...,-0.003522,0.019442,1.0,False,0.270136,0.498392,0.219513,False,supervised_popularity_comparator,primary
0,ASSISTments,problem,all_supported,attempted,popularity_discovery_early,all,10,0.011474,0.007234,0.011900,...,0.000000,0.000000,1.0,False,0.270136,0.498392,0.219513,False,reference_baseline,primary
18,ASSISTments,problem,all_supported,attempted,content_problem,all,10,0.007038,0.003517,0.007825,...,-0.003717,-0.004075,704.0,False,0.270136,0.498392,0.219513,False,does_not_pass_phase3_gate,primary
21,ASSISTments,skill,all_supported,attempted,learner_neighbor_cf,all,10,0.463514,0.689105,0.734290,...,0.428747,0.478706,15.7,True,0.689105,0.734290,0.975155,False,eligible_for_phase4,secondary
12,ASSISTments,skill,all_supported,attempted,same_cluster_neighbor_cf,all,10,0.450994,0.668541,0.713161,...,0.408183,0.457576,15.5,True,0.689105,0.734290,0.975155,False,exclude_cluster_signal,secondary
25,ASSISTments,skill,all_supported,attempted,sequential_transition,all,10,0.248878,0.432834,0.489179,...,0.172476,0.233595,15.7,True,0.689105,0.734290,0.975155,False,eligible_for_phase4,secondary


## Step 3 — Freeze tasks, targets, policies, segments and metrics


In [6]:
RELEVANCE_DEFINITIONS = {
    'attempted': 'relevance_binary',
    'successful': 'successful_future_item',
}
CANDIDATE_POLICIES = {
    'all_supported': {
        'exclude_seen': False,
        'description': 'Allow supported previously seen items; repeated practice is valid.',
    },
    'novel_only': {
        'exclude_seen': True,
        'description': (
            'Exclude items present in the learner early history and '
            'relevance denominator.'
        ),
    },
}
SEGMENTS = ('all', 'cold_start', 'non_cold_start')
TASK_DEFINITIONS = [
    {
        'Dataset': 'ASSISTments', 'Task': 'problem', 'Priority': 'primary',
        'CatalogFile': 'problem_catalog.parquet',
        'EarlyHistoryFile': 'early_problem_history.parquet',
        'FutureRelevanceFile': 'future_problem_relevance.parquet',
        'EvaluableColumn': 'evaluable_problem',
        'ColdStartColumn': 'cold_start_problem_history',
        'ClusterUse': 'accepted_ablation',
    },
    {
        'Dataset': 'ASSISTments', 'Task': 'skill', 'Priority': 'secondary',
        'CatalogFile': 'skill_catalog.parquet',
        'EarlyHistoryFile': 'early_skill_history.parquet',
        'FutureRelevanceFile': 'future_skill_relevance.parquet',
        'EvaluableColumn': 'evaluable_skill',
        'ColdStartColumn': 'cold_start_skill_history',
        'ClusterUse': 'accepted_ablation',
    },
    {
        'Dataset': 'KDD', 'Task': 'problem', 'Priority': 'external_validation',
        'CatalogFile': 'problem_catalog.parquet',
        'EarlyHistoryFile': 'early_problem_history.parquet',
        'FutureRelevanceFile': 'future_problem_relevance.parquet',
        'EvaluableColumn': 'evaluable_problem',
        'ColdStartColumn': 'cold_start_problem_history',
        'ClusterUse': 'exploratory_ablation_only',
    },
    {
        'Dataset': 'KDD', 'Task': 'skill', 'Priority': 'external_validation',
        'CatalogFile': 'skill_catalog.parquet',
        'EarlyHistoryFile': 'early_skill_history.parquet',
        'FutureRelevanceFile': 'future_skill_relevance.parquet',
        'EvaluableColumn': 'evaluable_skill',
        'ColdStartColumn': 'cold_start_skill_history',
        'ClusterUse': 'exploratory_ablation_only',
    },
]
task_table = pd.DataFrame(TASK_DEFINITIONS)
summary_lookup = benchmark_summary.set_index('Dataset')
task_table['CandidateCount'] = task_table.apply(
    lambda row: int(summary_lookup.loc[
        row['Dataset'],
        'ProblemCandidates' if row['Task'] == 'problem' else 'SkillCandidates',
    ]), axis=1,
)
task_table['ValidationEvaluableRate'] = task_table.apply(
    lambda row: float(summary_lookup.loc[
        row['Dataset'],
        'ValidationProblemEvaluableRate' if row['Task'] == 'problem'
        else 'ValidationSkillEvaluableRate',
    ]), axis=1,
)
task_table['ValidationColdStartRate'] = task_table.apply(
    lambda row: float(summary_lookup.loc[
        row['Dataset'],
        'ValidationProblemColdStartRate' if row['Task'] == 'problem'
        else 'ValidationSkillColdStartRate',
    ]), axis=1,
)
EXPERIMENT_CONTRACT = {
    'selection_source': 'discovery_only',
    'final_evaluation_cohort': 'validation',
    'selection_metric': 'NDCGAtK',
    'selection_k': PRIMARY_K,
    'selection_relevance': PRIMARY_RELEVANCE,
    'selection_candidate_policy': PRIMARY_CANDIDATE_POLICY,
    'selection_segment': PRIMARY_SEGMENT,
    'metric_ks': list(METRIC_KS),
    'segments': list(SEGMENTS),
    'candidate_policies': list(CANDIDATE_POLICIES),
    'relevance_definitions': RELEVANCE_DEFINITIONS,
    'novel_relevance_excludes_seen': True,
    'validation_future_role': 'final_metrics_and_bootstrap_only',
    'validation_reuse_status': 'previously_evaluated_for_phase3_component_gates',
    'inference_scope': 'frozen_cohort_post_selection_not_independent_test',
    'kdd_decision_authority': 'secondary_evidence_only',
    'dense_learner_item_matrix_constructed': False,
}
assert len(task_table) == 4
assert task_table['CandidateCount'].gt(MAX_RECOMMENDATIONS).all()
assert set(RELEVANCE_DEFINITIONS) == {'attempted', 'successful'}
assert set(CANDIDATE_POLICIES) == {'all_supported', 'novel_only'}
assert EXPERIMENT_CONTRACT['novel_relevance_excludes_seen'] is True
display(task_table)
display(pd.json_normalize(EXPERIMENT_CONTRACT, sep='.').T.rename(columns={0: 'Value'}))

,Dataset,Task,Priority,CatalogFile,EarlyHistoryFile,FutureRelevanceFile,EvaluableColumn,ColdStartColumn,ClusterUse,CandidateCount,ValidationEvaluableRate,ValidationColdStartRate
0,ASSISTments,problem,primary,problem_catalog.parquet,early_problem_history.parquet,future_problem_relevance.parquet,evaluable_problem,cold_start_problem_history,accepted_ablation,39779,0.899355,0.024599
1,ASSISTments,skill,secondary,skill_catalog.parquet,early_skill_history.parquet,future_skill_relevance.parquet,evaluable_skill,cold_start_skill_history,accepted_ablation,161,0.588271,0.375731
2,KDD,problem,external_validation,problem_catalog.parquet,early_problem_history.parquet,future_problem_relevance.parquet,evaluable_problem,cold_start_problem_history,exploratory_ablation_only,855,0.982301,0.000000
3,KDD,skill,external_validation,skill_catalog.parquet,early_skill_history.parquet,future_skill_relevance.parquet,evaluable_skill,cold_start_skill_history,exploratory_ablation_only,99,0.991150,0.000000


,Value
selection_source,discovery_only
final_evaluation_cohort,validation
selection_metric,NDCGAtK
selection_k,10
selection_relevance,attempted
selection_candidate_policy,all_supported
selection_segment,all
metric_ks,"[5, 10, 20]"
segments,"[all, cold_start, non_cold_start]"
candidate_policies,"[all_supported, novel_only]"


## Step 4 — Register the production hybrid and research ablations

Actual score normalization and fusion Step 5.

In [7]:
COMPONENT_REGISTRY = {
    'learner_neighbor_cf': {
        'Signal': 'collaborative', 'Phase3Gate': 'passed',
        'AllowedInProductionCandidate': True,
    },
    'sequential_transition': {
        'Signal': 'sequential', 'Phase3Gate': 'passed',
        'AllowedInProductionCandidate': True,
        'GenuineEvidenceFilter': "score_source == 'transition'",
    },
    'content_problem': {
        'Signal': 'content', 'Phase3Gate': 'failed_assistments',
        'AllowedInProductionCandidate': False,
    },
    'weak_skill_mastery_confidence': {
        'Signal': 'weakness', 'Phase3Gate': 'failed_assistments',
        'AllowedInProductionCandidate': False,
    },
    'same_cluster_neighbor_cf': {
        'Signal': 'cluster_neighbor', 'Phase3Gate': 'failed_assistments',
        'AllowedInProductionCandidate': False,
    },
    'cluster_popularity': {
        'Signal': 'cluster_popularity', 'Phase3Gate': 'failed_assistments',
        'AllowedInProductionCandidate': False,
    },
    'popularity_discovery_early': {
        'Signal': 'fallback', 'Phase3Gate': 'reference_baseline',
        'AllowedInProductionCandidate': False,
    },
}
VARIANT_TEMPLATES = [
    {
        'Variant': 'hybrid_neighbor_sequential',
        'ApplicableTask': 'all',
        'Components': ('learner_neighbor_cf', 'sequential_transition'),
        'Role': 'production_candidate',
    },
    {
        'Variant': 'hybrid_problem_with_content',
        'ApplicableTask': 'problem',
        'Components': (
            'learner_neighbor_cf', 'sequential_transition', 'content_problem',
        ),
        'Role': 'research_ablation',
    },
    {
        'Variant': 'hybrid_skill_with_weakness',
        'ApplicableTask': 'skill',
        'Components': (
            'learner_neighbor_cf', 'sequential_transition',
            'weak_skill_mastery_confidence',
        ),
        'Role': 'research_ablation',
    },
    {
        'Variant': 'hybrid_with_same_cluster_neighbor',
        'ApplicableTask': 'all',
        'Components': (
            'learner_neighbor_cf', 'sequential_transition',
            'same_cluster_neighbor_cf',
        ),
        'Role': 'cluster_ablation',
    },
    {
        'Variant': 'hybrid_with_cluster_popularity',
        'ApplicableTask': 'all',
        'Components': (
            'learner_neighbor_cf', 'sequential_transition',
            'cluster_popularity',
        ),
        'Role': 'cluster_ablation',
    },
]
variant_rows = []
for task_definition in TASK_DEFINITIONS:
    for template in VARIANT_TEMPLATES:
        if template['ApplicableTask'] not in {'all', task_definition['Task']}:
            continue
        variant_rows.append({
            'Dataset': task_definition['Dataset'],
            'Task': task_definition['Task'],
            'Priority': task_definition['Priority'],
            'Variant': template['Variant'],
            'Components': json.dumps(template['Components']),
            'Role': template['Role'],
            'CanSelectProductionModel': (
                task_definition['Dataset'] == PRIMARY_DATASET
                and task_definition['Task'] == PRIMARY_TASK
                and template['Role'] in {'production_candidate', 'cluster_ablation'}
            ),
            'KDDInterpretation': (
                'secondary_evidence_only'
                if task_definition['Dataset'] == 'KDD' else 'not_applicable'
            ),
        })
hybrid_variant_table = pd.DataFrame(variant_rows)
production_components = tuple(
    json.loads(hybrid_variant_table.loc[
        hybrid_variant_table['Variant'].eq('hybrid_neighbor_sequential'),
        'Components',
    ].iloc[0])
)
assert production_components == (
    'learner_neighbor_cf', 'sequential_transition',
)
assert all(
    COMPONENT_REGISTRY[name]['AllowedInProductionCandidate']
    for name in production_components
)
non_primary_task = ~(
    hybrid_variant_table['Dataset'].eq(PRIMARY_DATASET)
    & hybrid_variant_table['Task'].eq(PRIMARY_TASK)
)
assert not hybrid_variant_table.loc[
    non_primary_task, 'CanSelectProductionModel'
].any()
assert set(hybrid_variant_table.loc[
    hybrid_variant_table['CanSelectProductionModel'], 'Variant'
]) == {
    'hybrid_neighbor_sequential',
    'hybrid_with_same_cluster_neighbor',
    'hybrid_with_cluster_popularity',
}
assert hybrid_variant_table.loc[
    hybrid_variant_table['Role'].eq('production_candidate')
].groupby(['Dataset', 'Task']).size().eq(1).all()
assert set(hybrid_variant_table.loc[
    hybrid_variant_table['Role'].eq('cluster_ablation'), 'Variant'
]) == {
    'hybrid_with_same_cluster_neighbor',
    'hybrid_with_cluster_popularity',
}
component_registry_table = (
    pd.DataFrame.from_dict(COMPONENT_REGISTRY, orient='index')
    .rename_axis('Component').reset_index()
)
display(component_registry_table)
display(hybrid_variant_table.sort_values(['Dataset', 'Task', 'Role', 'Variant']))

,Component,Signal,Phase3Gate,AllowedInProductionCandidate,GenuineEvidenceFilter
0,learner_neighbor_cf,collaborative,passed,True,NaN
1,sequential_transition,sequential,passed,True,score_source == 'transition'
2,content_problem,content,failed_assistments,False,NaN
3,weak_skill_mastery_confidence,weakness,failed_assistments,False,NaN
4,same_cluster_neighbor_cf,cluster_neighbor,failed_assistments,False,NaN
5,cluster_popularity,cluster_popularity,failed_assistments,False,NaN
6,popularity_discovery_early,fallback,reference_baseline,False,NaN


,Dataset,Task,Priority,Variant,Components,Role,CanSelectProductionModel,KDDInterpretation
3,ASSISTments,problem,primary,hybrid_with_cluster_popularity,"[""learner_neighbor_cf"", ""sequential_transition...",cluster_ablation,True,not_applicable
2,ASSISTments,problem,primary,hybrid_with_same_cluster_neighbor,"[""learner_neighbor_cf"", ""sequential_transition...",cluster_ablation,True,not_applicable
0,ASSISTments,problem,primary,hybrid_neighbor_sequential,"[""learner_neighbor_cf"", ""sequential_transition""]",production_candidate,True,not_applicable
1,ASSISTments,problem,primary,hybrid_problem_with_content,"[""learner_neighbor_cf"", ""sequential_transition...",research_ablation,False,not_applicable
7,ASSISTments,skill,secondary,hybrid_with_cluster_popularity,"[""learner_neighbor_cf"", ""sequential_transition...",cluster_ablation,False,not_applicable
6,ASSISTments,skill,secondary,hybrid_with_same_cluster_neighbor,"[""learner_neighbor_cf"", ""sequential_transition...",cluster_ablation,False,not_applicable
4,ASSISTments,skill,secondary,hybrid_neighbor_sequential,"[""learner_neighbor_cf"", ""sequential_transition""]",production_candidate,False,not_applicable
5,ASSISTments,skill,secondary,hybrid_skill_with_weakness,"[""learner_neighbor_cf"", ""sequential_transition...",research_ablation,False,not_applicable
11,KDD,problem,external_validation,hybrid_with_cluster_popularity,"[""learner_neighbor_cf"", ""sequential_transition...",cluster_ablation,False,secondary_evidence_only
10,KDD,problem,external_validation,hybrid_with_same_cluster_neighbor,"[""learner_neighbor_cf"", ""sequential_transition...",cluster_ablation,False,secondary_evidence_only


## Step 5 — Comparable sparse component ranks and weighted fusion

Converts each component's sparse top-20 list into reciprocal-rank evidence, combines only positive-weight evidence and retains component-level ranks and contributions. Sequential popularity fallback is excluded from sequential evidence.

In [8]:
from itertools import product
import gc

import pyarrow as pa
import pyarrow.dataset as pds
from scipy.sparse import csr_matrix, diags
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

FUSION_COMPONENTS = (
    'learner_neighbor_cf', 'sequential_transition', 'content_problem',
    'weak_skill_mastery_confidence', 'same_cluster_neighbor_cf',
    'cluster_popularity',
)
FALLBACK_COMPONENT = 'popularity_discovery_early'
WEIGHT_GRID_STEP = 0.25

def deterministic_top_k(scored, max_k=MAX_RECOMMENDATIONS):
    required = {'learner_id', 'item_id', 'score'}
    if not required.issubset(scored.columns):
        missing = sorted(required - set(scored.columns))
        raise ValueError(f'Missing recommendation columns: {missing}')
    if scored.empty:
        return pd.DataFrame(columns=['learner_id', 'item_id', 'score', 'rank'])
    ranked = scored[['learner_id', 'item_id', 'score']].copy()
    ranked['learner_id'] = ranked['learner_id'].astype(str)
    ranked['item_id'] = ranked['item_id'].astype(str)
    ranked['score'] = pd.to_numeric(ranked['score'], errors='raise').astype(float)
    assert np.isfinite(ranked['score']).all()
    ranked = ranked.sort_values(
        ['learner_id', 'score', 'item_id'],
        ascending=[True, False, True], kind='mergesort',
    ).drop_duplicates(['learner_id', 'item_id'], keep='first')
    ranked = ranked.groupby('learner_id', sort=False).head(max_k).copy()
    ranked['rank'] = ranked.groupby('learner_id', sort=False).cumcount() + 1
    return ranked

def standardise_component_recommendations(
    recommendations, component, dataset, task, default_score_source=None,
):
    required = {'learner_id', 'item_id', 'score', 'rank'}
    if not required.issubset(recommendations.columns):
        missing = sorted(required - set(recommendations.columns))
        raise ValueError(f'{component} is missing {missing}')
    frame = recommendations.copy()
    frame['learner_id'] = frame['learner_id'].astype(str)
    frame['item_id'] = frame['item_id'].astype(str)
    frame['score'] = pd.to_numeric(frame['score'], errors='raise').astype(float)
    frame['rank'] = pd.to_numeric(frame['rank'], errors='raise').astype('int32')
    if component == 'sequential_transition' and 'score_source' in frame.columns:
        frame = frame[frame['score_source'].eq('transition')].copy()
    if 'score_source' not in frame.columns:
        frame['score_source'] = default_score_source or component
    else:
        frame['score_source'] = frame['score_source'].fillna(
            default_score_source or component
        ).astype(str)
    frame = frame[frame['rank'].between(1, MAX_RECOMMENDATIONS)].copy()
    assert not frame.duplicated(['learner_id', 'item_id']).any()
    assert np.isfinite(frame['score']).all()
    frame.insert(0, 'Component', component)
    frame.insert(0, 'Task', task)
    frame.insert(0, 'Dataset', dataset)
    return frame[[
        'Dataset', 'Task', 'Component', 'learner_id', 'item_id',
        'score', 'rank', 'score_source',
    ]]

def simplex_weight_grid(components, step=WEIGHT_GRID_STEP):
    components = tuple(components)
    units = int(round(1.0 / step))
    if not np.isclose(units * step, 1.0):
        raise ValueError('Weight-grid step must divide 1.0 exactly.')
    grid = []
    for allocation in product(range(units + 1), repeat=len(components)):
        if sum(allocation) != units:
            continue
        grid.append({
            component: count / units
            for component, count in zip(components, allocation)
        })
    assert grid
    assert all(np.isclose(sum(weights.values()), 1.0) for weights in grid)
    return grid

def weight_signature(components, weights):
    return '|'.join(
        f'{component}={float(weights.get(component, 0.0)):.2f}'
        for component in components
    )

def fuse_ranked_components(
    component_rows, components, weights, query_learner_ids,
    popularity_rows, max_k=MAX_RECOMMENDATIONS, rrf_constant=RRF_CONSTANT,
):
    components = tuple(components)
    if set(weights) != set(components):
        raise ValueError('Weights must match the variant components exactly.')
    if not np.isclose(sum(weights.values()), 1.0):
        raise ValueError('Fusion weights must sum to one.')
    query_ids = sorted(pd.Series(query_learner_ids, dtype='string').dropna().astype(str).unique())
    active = component_rows[component_rows['Component'].isin(components)].copy()
    if not active.empty:
        active['learner_id'] = active['learner_id'].astype(str)
        active['item_id'] = active['item_id'].astype(str)
        active = active[active['learner_id'].isin(query_ids)].copy()
        active['ComponentRank'] = pd.to_numeric(active['rank'], errors='raise').astype(float)
        active['ComponentWeight'] = active['Component'].map(weights).astype(float)
        active['Contribution'] = (
            active['ComponentWeight']
            / (float(rrf_constant) + active['ComponentRank'])
        )
        active = active[active['Contribution'].gt(0)].copy()

    if active.empty:
        fused = pd.DataFrame(columns=['learner_id', 'item_id', 'score', 'score_source'])
    else:
        keys = ['learner_id', 'item_id']
        fused = active.groupby(keys, sort=False)['Contribution'].sum().rename('score').reset_index()
        provenance = active.groupby(keys, sort=False)['Component'].agg(
            lambda values: '|'.join(sorted(set(values)))
        ).rename('score_source').reset_index()
        fused = fused.merge(provenance, on=keys, how='left', validate='one_to_one')
        rank_wide = active.pivot_table(
            index=keys, columns='Component', values='ComponentRank', aggfunc='min'
        ).rename(columns=lambda name: f'rank__{name}').reset_index()
        contribution_wide = active.pivot_table(
            index=keys, columns='Component', values='Contribution', aggfunc='sum', fill_value=0.0
        ).rename(columns=lambda name: f'contribution__{name}').reset_index()
        fused = fused.merge(rank_wide, on=keys, how='left', validate='one_to_one')
        fused = fused.merge(contribution_wide, on=keys, how='left', validate='one_to_one')
        fused = fused.sort_values(
            ['learner_id', 'score', 'item_id'],
            ascending=[True, False, True], kind='mergesort',
        ).groupby('learner_id', sort=False).head(max_k).copy()
        fused['rank'] = fused.groupby('learner_id', sort=False).cumcount() + 1
        fused['FallbackUsed'] = False

    recipients = set(fused['learner_id'].astype(str)) if len(fused) else set()
    missing_ids = sorted(set(query_ids) - recipients)
    fallback = popularity_rows[
        popularity_rows['learner_id'].astype(str).isin(missing_ids)
    ].copy()
    if len(fallback):
        fallback['learner_id'] = fallback['learner_id'].astype(str)
        fallback['item_id'] = fallback['item_id'].astype(str)
        fallback = fallback.sort_values(
            ['learner_id', 'rank', 'item_id'], kind='mergesort'
        ).groupby('learner_id', sort=False).head(max_k).copy()
        fallback['score'] = -pd.to_numeric(fallback['rank'], errors='raise').astype(float)
        fallback['score_source'] = 'popularity_fallback'
        fallback['FallbackUsed'] = True
        fallback = fallback[[
            'learner_id', 'item_id', 'score', 'rank',
            'score_source', 'FallbackUsed',
        ]]

    for component in FUSION_COMPONENTS:
        rank_column = f'rank__{component}'
        contribution_column = f'contribution__{component}'
        if rank_column not in fused.columns:
            fused[rank_column] = np.nan
        if contribution_column not in fused.columns:
            fused[contribution_column] = 0.0
        if len(fallback):
            fallback[rank_column] = np.nan
            fallback[contribution_column] = 0.0

    columns = [
        'learner_id', 'item_id', 'score', 'rank', 'score_source', 'FallbackUsed',
    ] + [f'rank__{component}' for component in FUSION_COMPONENTS] + [
        f'contribution__{component}' for component in FUSION_COMPONENTS
    ]
    parts = [fused[columns]] if len(fused) else []
    if len(fallback):
        parts.append(fallback[columns])
    result = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=columns)
    result['WeightSignature'] = weight_signature(components, weights)
    result = result.sort_values(
        ['learner_id', 'rank', 'item_id'], kind='mergesort'
    ).reset_index(drop=True)
    assert set(result['learner_id'].astype(str)).issubset(set(query_ids))
    assert not result.duplicated(['learner_id', 'item_id']).any()
    assert result.groupby('learner_id').size().le(max_k).all()
    assert np.isfinite(pd.to_numeric(result['score'], errors='coerce')).all()
    return result

assert len(simplex_weight_grid(('a', 'b'))) == 5
assert len(simplex_weight_grid(('a', 'b', 'c'))) == 15

## Step 6 — Recreate discovery-only tuning component recommendations

Recreate sparse top-20 predictions for the discovery-training/discovery-tuning split. Collaborative targets use discovery-training future labels. Content and weakness use tuning learners' early profiles. Sequential scores reuse discovery-early transitions plus the compact discovery recent-source artifact saved by Phase 3.

In [9]:
NEIGHBOR_QUERY_BATCH_SIZE = 512
WEAK_SKILL_STATES = {'weak', 'developing'}
CONTENT_WEIGHTS = {
    'weak_skill_alignment': 0.65,
    'difficulty_suitability': 0.20,
    'problem_type_affinity': 0.10,
    'hierarchy_affinity': 0.05,
}
POPULARITY_TIE_BREAK_WEIGHT = 1e-9

def load_task_tables(task_definition):
    directory = DATASET_DIRS[task_definition['Dataset']]
    return {
        'learners': pd.read_parquet(directory / 'learner_splits.parquet'),
        'catalog': pd.read_parquet(directory / task_definition['CatalogFile']),
        'early_history': pd.read_parquet(directory / task_definition['EarlyHistoryFile']),
        'future_relevance': pd.read_parquet(directory / task_definition['FutureRelevanceFile']),
    }

def build_sparse_history_matrix(history, learner_order, item_order):
    learner_order = [str(value) for value in learner_order]
    item_order = [str(value) for value in item_order]
    learner_index = {value: index for index, value in enumerate(learner_order)}
    item_index = {value: index for index, value in enumerate(item_order)}
    subset = history[
        history['learner_id'].astype(str).isin(learner_index)
        & history['item_id'].astype(str).isin(item_index)
        & history['in_candidate_catalog']
    ][['learner_id', 'item_id', 'early_interaction_count']].copy()
    rows = subset['learner_id'].astype(str).map(learner_index).to_numpy()
    columns = subset['item_id'].astype(str).map(item_index).to_numpy()
    values = np.log1p(pd.to_numeric(
        subset['early_interaction_count'], errors='coerce'
    ).fillna(0).to_numpy(float))
    matrix = csr_matrix(
        (values, (rows, columns)),
        shape=(len(learner_order), len(item_order)), dtype=np.float32,
    )
    matrix.eliminate_zeros()
    return matrix

def build_target_matrix(relevance, learner_order, item_order, relevance_column):
    learner_order = [str(value) for value in learner_order]
    item_order = [str(value) for value in item_order]
    learner_index = {value: index for index, value in enumerate(learner_order)}
    item_index = {value: index for index, value in enumerate(item_order)}
    labels = relevance[
        relevance['learner_id'].astype(str).isin(learner_index)
        & relevance['item_id'].astype(str).isin(item_index)
        & relevance['in_candidate_catalog']
        & relevance[relevance_column].eq(1)
    ][['learner_id', 'item_id']].drop_duplicates()
    rows = labels['learner_id'].astype(str).map(learner_index).to_numpy()
    columns = labels['item_id'].astype(str).map(item_index).to_numpy()
    return csr_matrix(
        (np.ones(len(labels), dtype=np.float32), (rows, columns)),
        shape=(len(learner_order), len(item_order)), dtype=np.float32,
    )

def query_neighbors(training_matrix, query_matrix, neighbor_count):
    if training_matrix.shape[0] == 0:
        raise ValueError('Cannot fit neighbours without discovery-training learners.')
    usable = min(int(neighbor_count), training_matrix.shape[0])
    model = NearestNeighbors(
        n_neighbors=usable, metric='cosine', algorithm='brute', n_jobs=-1,
    )
    model.fit(training_matrix)
    distances, indices = model.kneighbors(query_matrix)
    similarities = np.clip(1.0 - distances, 0.0, 1.0).astype(np.float32)
    return indices, similarities

def aggregate_neighbor_scores(
    query_ids, target_matrix, neighbor_indices, similarities, item_order,
    max_k=MAX_RECOMMENDATIONS, batch_size=NEIGHBOR_QUERY_BATCH_SIZE,
):
    query_ids = [str(value) for value in query_ids]
    item_order = [str(value) for value in item_order]
    rows = []
    training_count = target_matrix.shape[0]
    for batch_start in range(0, len(query_ids), batch_size):
        batch_end = min(batch_start + batch_size, len(query_ids))
        batch_indices = neighbor_indices[batch_start:batch_end]
        batch_similarities = similarities[batch_start:batch_end]
        local_rows = np.repeat(np.arange(batch_end - batch_start), batch_indices.shape[1])
        weights = csr_matrix(
            (batch_similarities.ravel(), (local_rows, batch_indices.ravel())),
            shape=(batch_end - batch_start, training_count), dtype=np.float32,
        )
        weights.eliminate_zeros()
        totals = np.asarray(weights.sum(axis=1)).ravel()
        inverse = np.divide(1.0, totals, out=np.zeros_like(totals), where=totals > 0)
        scored = (diags(inverse) @ weights @ target_matrix).tocsr()
        scored.eliminate_zeros()
        for local_index, learner_id in enumerate(query_ids[batch_start:batch_end]):
            start, end = scored.indptr[local_index], scored.indptr[local_index + 1]
            candidates = [
                (float(value), item_order[item_index])
                for item_index, value in zip(scored.indices[start:end], scored.data[start:end])
                if value > 0
            ]
            candidates.sort(key=lambda value: (-value[0], value[1]))
            for rank, (score, item_id) in enumerate(candidates[:max_k], start=1):
                rows.append((learner_id, item_id, score, rank))
    return pd.DataFrame(rows, columns=['learner_id', 'item_id', 'score', 'rank'])

def learner_neighbor_tuning_recommendations(
    tables, training_ids, query_ids, neighbor_count, item_order,
):
    training_matrix = build_sparse_history_matrix(tables['early_history'], training_ids, item_order)
    query_matrix = build_sparse_history_matrix(tables['early_history'], query_ids, item_order)
    indices, similarities = query_neighbors(training_matrix, query_matrix, neighbor_count)
    target_matrix = build_target_matrix(
        tables['future_relevance'], training_ids, item_order,
        RELEVANCE_DEFINITIONS['attempted'],
    )
    recommendations = aggregate_neighbor_scores(
        query_ids, target_matrix, indices, similarities, item_order,
    )
    del training_matrix, query_matrix, indices, similarities, target_matrix
    return recommendations

def same_cluster_neighbor_tuning_recommendations(
    tables, training_ids, query_ids, neighbor_count, item_order,
):
    learners = tables['learners'].copy()
    training_set, query_set = set(training_ids), set(query_ids)
    parts = []
    for cluster_value in sorted(learners['cluster'].dropna().unique(), key=str):
        cluster_training = sorted(learners.loc[
            learners['learner_id'].astype(str).isin(training_set)
            & learners['cluster'].eq(cluster_value), 'learner_id'
        ].astype(str))
        cluster_query = sorted(learners.loc[
            learners['learner_id'].astype(str).isin(query_set)
            & learners['cluster'].eq(cluster_value), 'learner_id'
        ].astype(str))
        if not cluster_training or not cluster_query:
            continue
        part = learner_neighbor_tuning_recommendations(
            tables, cluster_training, cluster_query, neighbor_count, item_order,
        )
        if len(part):
            parts.append(part)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
        columns=['learner_id', 'item_id', 'score', 'rank']
    )

def cluster_popularity_tuning_recommendations(
    tables, training_ids, query_ids, relevance_column='relevance_binary',
):
    learners = tables['learners'][['learner_id', 'cluster']].copy()
    learners['learner_id'] = learners['learner_id'].astype(str)
    training = learners[learners['learner_id'].isin(set(training_ids))]
    query = learners[learners['learner_id'].isin(set(query_ids))]
    labels = tables['future_relevance'][
        tables['future_relevance']['learner_id'].astype(str).isin(set(training_ids))
        & tables['future_relevance']['in_candidate_catalog']
        & tables['future_relevance'][relevance_column].eq(1)
    ][['learner_id', 'item_id']].drop_duplicates()
    labels['learner_id'] = labels['learner_id'].astype(str)
    labels['item_id'] = labels['item_id'].astype(str)
    counts = labels.merge(training, on='learner_id', how='inner').groupby(
        ['cluster', 'item_id']
    )['learner_id'].nunique().rename('score').reset_index()
    parts = []
    for cluster_value, query_group in query.groupby('cluster'):
        ranking = counts[counts['cluster'].eq(cluster_value)].sort_values(
            ['score', 'item_id'], ascending=[False, True], kind='mergesort'
        ).head(MAX_RECOMMENDATIONS)
        for learner_id in sorted(query_group['learner_id']):
            part = ranking[['item_id', 'score']].copy()
            part.insert(0, 'learner_id', learner_id)
            part['rank'] = np.arange(1, len(part) + 1)
            parts.append(part)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
        columns=['learner_id', 'item_id', 'score', 'rank']
    )

def content_tuning_recommendations(task_definition, tables, query_ids, tolerance):
    directory = DATASET_DIRS[task_definition['Dataset']]
    skill_history = pd.read_parquet(directory / 'early_skill_history.parquet')
    problem_history = tables['early_history']
    catalog = tables['catalog']
    problem_skill_map = pd.read_parquet(directory / 'problem_skill_map.parquet')
    query_set = set(query_ids)
    needs = skill_history[
        skill_history['learner_id'].astype(str).isin(query_set)
        & skill_history['in_candidate_catalog']
        & skill_history['skill_state'].isin(WEAK_SKILL_STATES)
    ][['learner_id', 'item_id', 'empirical_bayes_mastery', 'mastery_evidence_confidence']].copy()
    needs['learner_id'] = needs['learner_id'].astype(str)
    needs['item_id'] = needs['item_id'].astype(str)
    needs['weakness'] = (
        (1.0 - needs['empirical_bayes_mastery'])
        * needs['mastery_evidence_confidence']
    )
    mapping = problem_skill_map[['problem_item_id', 'skill_item_id', 'association_share']].copy()
    mapping['problem_item_id'] = mapping['problem_item_id'].astype(str)
    mapping['skill_item_id'] = mapping['skill_item_id'].astype(str)
    mapped = needs.merge(mapping, left_on='item_id', right_on='skill_item_id', how='inner')
    mapped['alignment'] = mapped['weakness'] * mapped['association_share']
    mapped['mastery_part'] = mapped['empirical_bayes_mastery'] * mapped['association_share']
    scored = mapped.groupby(['learner_id', 'problem_item_id'], sort=False).agg(
        weak_skill_alignment=('alignment', 'sum'),
        weighted_mastery=('mastery_part', 'sum'),
        association_coverage=('association_share', 'sum'),
    ).reset_index().rename(columns={'problem_item_id': 'item_id'})
    scored['profile_mastery'] = (
        scored['weighted_mastery'] / scored['association_coverage'].clip(lower=1e-12)
    ).clip(0, 1)
    metadata = catalog[[
        'item_id', 'difficulty_proxy', 'problem_type',
        'hierarchy', 'training_popularity',
    ]].copy()
    metadata['item_id'] = metadata['item_id'].astype(str)
    metadata['difficulty_proxy'] = metadata['difficulty_proxy'].fillna(0.5).clip(0, 1)
    metadata['problem_type'] = metadata['problem_type'].fillna('__missing__').astype(str)
    metadata['hierarchy'] = metadata['hierarchy'].fillna('__missing__').astype(str)
    profiles = problem_history[
        problem_history['learner_id'].astype(str).isin(query_set)
        & problem_history['in_candidate_catalog']
    ][['learner_id', 'item_id', 'early_interaction_count']].copy()
    profiles['learner_id'] = profiles['learner_id'].astype(str)
    profiles['item_id'] = profiles['item_id'].astype(str)
    profiles = profiles.merge(
        metadata[['item_id', 'problem_type', 'hierarchy']],
        on='item_id', how='inner',
    )
    totals = (
        profiles.groupby('learner_id')['early_interaction_count']
        .transform('sum').clip(lower=1)
    )
    profiles['share'] = profiles['early_interaction_count'] / totals
    type_affinity = (
        profiles.groupby(['learner_id', 'problem_type'])['share'].sum()
        .rename('problem_type_affinity').reset_index()
    )
    hierarchy_affinity = (
        profiles.groupby(['learner_id', 'hierarchy'])['share'].sum()
        .rename('hierarchy_affinity').reset_index()
    )
    scored = scored.merge(metadata, on='item_id', how='inner').merge(
        type_affinity, on=['learner_id', 'problem_type'], how='left'
    ).merge(hierarchy_affinity, on=['learner_id', 'hierarchy'], how='left')
    scored[['problem_type_affinity', 'hierarchy_affinity']] = scored[[
        'problem_type_affinity', 'hierarchy_affinity'
    ]].fillna(0.0)
    scored['difficulty_suitability'] = (
        1.0 - (scored['difficulty_proxy'] - scored['profile_mastery']).abs() / float(tolerance)
    ).clip(0, 1)
    scored['score'] = (
        CONTENT_WEIGHTS['weak_skill_alignment'] * scored['weak_skill_alignment'].clip(0, 1)
        + CONTENT_WEIGHTS['difficulty_suitability'] * scored['difficulty_suitability']
        + CONTENT_WEIGHTS['problem_type_affinity'] * scored['problem_type_affinity']
        + CONTENT_WEIGHTS['hierarchy_affinity'] * scored['hierarchy_affinity']
        + POPULARITY_TIE_BREAK_WEIGHT * scored['training_popularity']
    )
    return deterministic_top_k(scored[['learner_id', 'item_id', 'score']])

def weakness_tuning_recommendations(tables, query_ids):
    query_set = set(query_ids)
    scored = tables['early_history'][
        tables['early_history']['learner_id'].astype(str).isin(query_set)
        & tables['early_history']['in_candidate_catalog']
        & tables['early_history']['skill_state'].isin(WEAK_SKILL_STATES)
    ][['learner_id', 'item_id', 'empirical_bayes_mastery', 'mastery_evidence_confidence']].copy()
    scored['score'] = (
        (1.0 - scored['empirical_bayes_mastery'])
        * scored['mastery_evidence_confidence']
    )
    return deterministic_top_k(scored[['learner_id', 'item_id', 'score']])

def sequential_tuning_recommendations(recent, transitions, decay, shrinkage):
    adjusted = transitions[[
        'source_item_id', 'target_item_id',
        'transition_count', 'source_transition_total',
    ]].copy()
    adjusted['transition_probability'] = adjusted['transition_count'] / (
        adjusted['source_transition_total'] + float(shrinkage)
    )
    joined = recent.merge(
        adjusted[['source_item_id', 'target_item_id', 'transition_probability']],
        on='source_item_id', how='inner', validate='many_to_many',
    )
    joined['weighted'] = (
        np.power(float(decay), joined['recency_rank'].astype(float))
        * joined['transition_probability']
    )
    scored = (
        joined.groupby(
            ['learner_id', 'target_item_id'], sort=False
        )['weighted'].sum().rename('score').reset_index().rename(
            columns={'target_item_id': 'item_id'}
        )
    )
    recommendations = deterministic_top_k(scored)
    recommendations['score_source'] = 'transition'
    return recommendations

def popularity_recommendations(catalog, query_ids):
    ranking = catalog[['item_id', 'training_interactions']].copy()
    ranking['item_id'] = ranking['item_id'].astype(str)
    ranking['score'] = ranking['training_interactions'].astype(float)
    ranking = ranking.sort_values(
        ['score', 'item_id'], ascending=[False, True], kind='mergesort'
    ).head(MAX_RECOMMENDATIONS)[['item_id', 'score']]
    parts = []
    for learner_id in sorted(query_ids):
        part = ranking.copy()
        part.insert(0, 'learner_id', str(learner_id))
        part['rank'] = np.arange(1, len(part) + 1)
        parts.append(part)
    return pd.concat(parts, ignore_index=True)

print('Defined leakage-safe sparse component builders for discovery-only tuning.')

Defined leakage-safe sparse component builders for discovery-only tuning.


In [10]:
SEQUENCE_STAGE_MANIFEST = BASELINE_ROOT / 'steps_11_12_artifact_manifest.csv'
SEQUENCE_QUERY_FILES = {
    'discovery_recent_items': BASELINE_ROOT / 'step11_discovery_recent_items.parquet',
}
assert SEQUENCE_STAGE_MANIFEST.is_file(), (
    'Phase 4 requires the compact Phase 3 Step 11 recent-source artifacts. '
    'Rerun Phase 3 Steps 11–12 if the staged manifest is missing.'
)
sequence_stage_manifest = pd.read_csv(SEQUENCE_STAGE_MANIFEST).set_index('Artifact')
for artifact, path in SEQUENCE_QUERY_FILES.items():
    assert path.is_file(), path
    manifest_row = sequence_stage_manifest.loc[artifact]
    assert path.name == manifest_row['File']
    assert pq.ParquetFile(path).metadata.num_rows == int(manifest_row['Rows'])
    assert path.stat().st_size == int(manifest_row['Bytes'])

TUNING_COMPONENT_PATH = OUTPUT_ROOT / 'step6_tuning_component_recommendations.parquet'
TUNING_SPLIT_PATH = OUTPUT_ROOT / 'step6_discovery_split.csv'
component_schema = pa.schema([
    pa.field('Dataset', pa.string()), pa.field('Task', pa.string()),
    pa.field('Component', pa.string()), pa.field('learner_id', pa.string()),
    pa.field('item_id', pa.string()), pa.field('score', pa.float64()),
    pa.field('rank', pa.int32()), pa.field('score_source', pa.string()),
])
writer = pq.ParquetWriter(TUNING_COMPONENT_PATH, component_schema, compression='snappy')
tuning_split_parts = []
tuning_component_rows = []
try:
    for task_definition in TASK_DEFINITIONS:
        dataset, task = task_definition['Dataset'], task_definition['Task']
        tables = load_task_tables(task_definition)
        for key in ['learners', 'catalog', 'early_history', 'future_relevance']:
            if 'learner_id' in tables[key].columns:
                tables[key]['learner_id'] = tables[key]['learner_id'].astype(str)
            if 'item_id' in tables[key].columns:
                tables[key]['item_id'] = tables[key]['item_id'].astype(str)
        discovery_ids = sorted(tables['learners'].loc[
            tables['learners']['cohort'].eq('discovery'), 'learner_id'
        ].astype(str))
        validation_ids = set(tables['learners'].loc[
            tables['learners']['cohort'].eq('validation'), 'learner_id'
        ].astype(str))
        discovery_train_ids, discovery_tuning_ids = train_test_split(
            discovery_ids, test_size=DISCOVERY_TUNING_FRACTION,
            random_state=RANDOM_STATE,
        )
        discovery_train_ids = sorted(discovery_train_ids)
        discovery_tuning_ids = sorted(discovery_tuning_ids)
        assert set(discovery_train_ids).isdisjoint(discovery_tuning_ids)
        assert set(discovery_tuning_ids).isdisjoint(validation_ids)
        split_part = pd.DataFrame({
            'Dataset': dataset, 'Task': task,
            'learner_id': discovery_train_ids + discovery_tuning_ids,
            'DiscoveryRole': (
                ['training'] * len(discovery_train_ids)
                + ['tuning'] * len(discovery_tuning_ids)
            ),
        })
        tuning_split_parts.append(split_part)
        item_order = sorted(tables['catalog']['item_id'].astype(str).unique())
        config_row = hyperparameter_selection[
            hyperparameter_selection['Dataset'].eq(dataset)
            & hyperparameter_selection['Task'].eq(task)
        ].iloc[0]
        neighbor_row = neighbor_configuration[
            neighbor_configuration['Dataset'].eq(dataset)
            & neighbor_configuration['Task'].eq(task)
        ].iloc[0]
        selected_neighbors = int(neighbor_row['SelectedNeighbors'])

        component_outputs = []
        ordinary_cf = learner_neighbor_tuning_recommendations(
            tables, discovery_train_ids, discovery_tuning_ids,
            selected_neighbors, item_order,
        )
        component_outputs.append(('learner_neighbor_cf', ordinary_cf, 'neighbor_cf'))

        recent = pd.read_parquet(
            SEQUENCE_QUERY_FILES['discovery_recent_items'],
            filters=[('Dataset', '==', dataset), ('Task', '==', task)],
        )
        recent['learner_id'] = recent['learner_id'].astype(str)
        recent['source_item_id'] = recent['source_item_id'].astype(str)
        recent = recent[recent['learner_id'].isin(discovery_tuning_ids)].copy()
        transitions = pd.read_parquet(
            BASELINE_ROOT / 'transition_probabilities.parquet',
            filters=[('Dataset', '==', dataset), ('Task', '==', task)],
        )
        transitions['source_item_id'] = transitions['source_item_id'].astype(str)
        transitions['target_item_id'] = transitions['target_item_id'].astype(str)
        sequential = sequential_tuning_recommendations(
            recent, transitions, float(config_row['SelectedRecencyDecay']),
            float(config_row['SelectedTransitionShrinkage']),
        )
        component_outputs.append(('sequential_transition', sequential, 'transition'))
        del recent, transitions

        if task == 'problem':
            content = content_tuning_recommendations(
                task_definition, tables, discovery_tuning_ids,
                float(config_row['SelectedDifficultyTolerance']),
            )
            component_outputs.append(('content_problem', content, 'content'))
        else:
            weakness = weakness_tuning_recommendations(tables, discovery_tuning_ids)
            component_outputs.append((
                'weak_skill_mastery_confidence', weakness, 'weakness',
            ))

        cluster_cf = same_cluster_neighbor_tuning_recommendations(
            tables, discovery_train_ids, discovery_tuning_ids,
            selected_neighbors, item_order,
        )
        component_outputs.append((
            'same_cluster_neighbor_cf', cluster_cf, 'same_cluster_neighbor',
        ))
        cluster_popularity = cluster_popularity_tuning_recommendations(
            tables, discovery_train_ids, discovery_tuning_ids,
        )
        component_outputs.append((
            'cluster_popularity', cluster_popularity, 'cluster_popularity',
        ))
        popularity = popularity_recommendations(tables['catalog'], discovery_tuning_ids)
        component_outputs.append((
            FALLBACK_COMPONENT, popularity, 'discovery_early_popularity',
        ))

        for component, recommendations, source in component_outputs:
            standard = standardise_component_recommendations(
                recommendations, component, dataset, task, source,
            )
            if len(standard):
                table = pa.Table.from_pandas(
                    standard, schema=component_schema, preserve_index=False,
                )
                writer.write_table(table)
            tuning_component_rows.append({
                'Dataset': dataset, 'Task': task, 'Component': component,
                'Rows': len(standard),
                'LearnersWithScores': standard['learner_id'].nunique(),
                'DiscoveryTrainingLearners': len(discovery_train_ids),
                'DiscoveryTuningLearners': len(discovery_tuning_ids),
                'ValidationLearnersUntouched': len(validation_ids),
            })
        del component_outputs, tables, ordinary_cf, sequential, cluster_cf
        globals().pop('content', None)
        globals().pop('weakness', None)
        gc.collect()
finally:
    writer.close()

tuning_split = pd.concat(tuning_split_parts, ignore_index=True)
tuning_split.to_csv(TUNING_SPLIT_PATH, index=False)
tuning_component_diagnostics = pd.DataFrame(tuning_component_rows)
assert set(tuning_split['DiscoveryRole']) == {'training', 'tuning'}
assert (
    pq.ParquetFile(TUNING_COMPONENT_PATH).metadata.num_rows
    == tuning_component_diagnostics['Rows'].sum()
)
assert tuning_component_diagnostics['ValidationLearnersUntouched'].gt(0).all()
display(tuning_component_diagnostics)
print('Saved discovery-only sparse tuning component recommendations.')

,Dataset,Task,Component,Rows,LearnersWithScores,DiscoveryTrainingLearners,DiscoveryTuningLearners,ValidationLearnersUntouched
0,ASSISTments,problem,learner_neighbor_cf,97422,5145,21334,5334,6667
1,ASSISTments,problem,sequential_transition,79093,5203,21334,5334,6667
2,ASSISTments,problem,content_problem,60189,3015,21334,5334,6667
3,ASSISTments,problem,same_cluster_neighbor_cf,97501,5133,21334,5334,6667
4,ASSISTments,problem,cluster_popularity,106680,5334,21334,5334,6667
5,ASSISTments,problem,popularity_discovery_early,106680,5334,21334,5334,6667
6,ASSISTments,skill,learner_neighbor_cf,51119,3263,21334,5334,6667
7,ASSISTments,skill,sequential_transition,65471,3291,21334,5334,6667
8,ASSISTments,skill,weak_skill_mastery_confidence,16511,3015,21334,5334,6667
9,ASSISTments,skill,same_cluster_neighbor_cf,53351,3256,21334,5334,6667


Saved discovery-only sparse tuning component recommendations.


## Step 7 — Discovery-only hybrid weight selection

Each variant searches a deterministic 0.25-simplex weight grid. Selection uses discovery-tuning attempted relevance, all-supported candidates, all evaluable learners and NDCG@10. Ties use Recall@10, catalogue coverage and then the lexical weight signature. Every tested configuration is saved.

In [11]:
def ranking_metrics_for_user(recommended_items, relevant_items, k):
    recommended = list(recommended_items[:k])
    relevant = set(relevant_items)
    if not relevant:
        return None
    hits = np.array([item in relevant for item in recommended], dtype=float)
    padded = np.pad(hits, (0, max(0, k - len(hits))))[:k]
    hit_count = padded.sum()
    discounts = np.log2(np.arange(2, k + 2))
    dcg = np.sum(padded / discounts)
    ideal_length = min(len(relevant), k)
    idcg = np.sum(np.ones(ideal_length) / discounts[:ideal_length])
    hit_positions = np.flatnonzero(padded)
    average_precision = sum(
        padded[:position + 1].sum() / (position + 1)
        for position in hit_positions
    ) / min(len(relevant), k)
    return {
        'PrecisionAtK': hit_count / k,
        'RecallAtK': hit_count / len(relevant),
        'NDCGAtK': dcg / idcg if idcg else 0.0,
        'MAPAtK': average_precision,
        'HitRateAtK': float(hit_count > 0),
    }

def evaluate_tuning_ranking(
    recommendations, future_relevance, learners, catalog, query_ids,
    evaluable_column, k=PRIMARY_K,
):
    query_ids = set(pd.Series(query_ids).astype(str))
    relevance = future_relevance.copy()
    relevance['learner_id'] = relevance['learner_id'].astype(str)
    relevance['item_id'] = relevance['item_id'].astype(str)
    relevant = relevance[
        relevance['learner_id'].isin(query_ids)
        & relevance['in_candidate_catalog']
        & relevance[RELEVANCE_DEFINITIONS['attempted']].eq(1)
    ][['learner_id', 'item_id']].drop_duplicates()
    relevant_by_user = relevant.groupby('learner_id')['item_id'].agg(set).to_dict()
    learner_frame = learners.copy()
    learner_frame['learner_id'] = learner_frame['learner_id'].astype(str)
    eligible = set(learner_frame.loc[
        learner_frame['learner_id'].isin(query_ids)
        & learner_frame[evaluable_column], 'learner_id'
    ]) & set(relevant_by_user)
    ranked = recommendations.sort_values(
        ['learner_id', 'rank', 'item_id'], kind='mergesort'
    )
    ranked_by_user = ranked.groupby('learner_id')['item_id'].agg(list).to_dict()
    user_rows = []
    recommendation_union = set()
    list_lengths = []
    for learner_id in sorted(eligible):
        items = ranked_by_user.get(learner_id, [])[:k]
        user_rows.append(ranking_metrics_for_user(items, relevant_by_user[learner_id], k))
        recommendation_union.update(items)
        list_lengths.append(len(items))
    if not user_rows:
        raise ValueError('No evaluable discovery-tuning learners for weight selection.')
    metrics = pd.DataFrame(user_rows).mean().to_dict()
    metrics.update({
        'CatalogCoverageAtK': len(recommendation_union) / len(catalog),
        'MeanRecommendations': float(np.mean(list_lengths)),
        'EvaluatedLearners': len(eligible),
        'FallbackLearners': recommendations.loc[
            recommendations['FallbackUsed'], 'learner_id'
        ].astype(str).nunique(),
    })
    return metrics

HYBRID_TUNING_METRICS_PATH = OUTPUT_ROOT / 'hybrid_tuning_metrics.csv'
HYBRID_WEIGHT_SELECTION_PATH = OUTPUT_ROOT / 'hybrid_weight_selection.csv'
tuning_metric_rows = []
selection_rows = []
tuning_split = pd.read_csv(TUNING_SPLIT_PATH, dtype={'learner_id': str})
for task_definition in TASK_DEFINITIONS:
    dataset, task = task_definition['Dataset'], task_definition['Task']
    tables = load_task_tables(task_definition)
    for key in ['learners', 'catalog', 'future_relevance']:
        if 'learner_id' in tables[key].columns:
            tables[key]['learner_id'] = tables[key]['learner_id'].astype(str)
        if 'item_id' in tables[key].columns:
            tables[key]['item_id'] = tables[key]['item_id'].astype(str)
    tuning_ids = sorted(tuning_split.loc[
        tuning_split['Dataset'].eq(dataset)
        & tuning_split['Task'].eq(task)
        & tuning_split['DiscoveryRole'].eq('tuning'), 'learner_id'
    ].astype(str))
    component_rows = pd.read_parquet(
        TUNING_COMPONENT_PATH,
        filters=[('Dataset', '==', dataset), ('Task', '==', task)],
    )
    component_rows['learner_id'] = component_rows['learner_id'].astype(str)
    component_rows['item_id'] = component_rows['item_id'].astype(str)
    popularity_rows = component_rows[
        component_rows['Component'].eq(FALLBACK_COMPONENT)
    ].copy()
    task_variants = hybrid_variant_table[
        hybrid_variant_table['Dataset'].eq(dataset)
        & hybrid_variant_table['Task'].eq(task)
    ]
    for variant_row in task_variants.itertuples(index=False):
        components = tuple(json.loads(variant_row.Components))
        variant_metrics = []
        for weights in simplex_weight_grid(components):
            recommendations = fuse_ranked_components(
                component_rows, components, weights, tuning_ids, popularity_rows,
            )
            metric_values = evaluate_tuning_ranking(
                recommendations, tables['future_relevance'], tables['learners'],
                tables['catalog'], tuning_ids, task_definition['EvaluableColumn'],
            )
            signature = weight_signature(components, weights)
            row = {
                'Dataset': dataset, 'Task': task, 'Variant': variant_row.Variant,
                'Role': variant_row.Role, 'WeightSignature': signature,
                'WeightsJSON': json.dumps(weights, sort_keys=True),
                'SelectionMetric': 'NDCGAtK', 'SelectionK': PRIMARY_K,
                'SelectionRelevance': PRIMARY_RELEVANCE,
                'SelectionCandidatePolicy': PRIMARY_CANDIDATE_POLICY,
                'SelectionSegment': PRIMARY_SEGMENT,
                'TuningSource': 'discovery_only',
                **metric_values,
            }
            tuning_metric_rows.append(row)
            variant_metrics.append(row)
        candidates = pd.DataFrame(variant_metrics).sort_values(
            ['NDCGAtK', 'RecallAtK', 'CatalogCoverageAtK', 'WeightSignature'],
            ascending=[False, False, False, True], kind='mergesort',
        )
        selected = candidates.iloc[0]
        selection_rows.append({
            'Dataset': dataset, 'Task': task, 'Variant': variant_row.Variant,
            'Role': variant_row.Role,
            'CanSelectProductionModel': bool(variant_row.CanSelectProductionModel),
            'SelectedWeightSignature': selected['WeightSignature'],
            'SelectedWeightsJSON': selected['WeightsJSON'],
            'SelectedNDCGAt10': selected['NDCGAtK'],
            'SelectedRecallAt10': selected['RecallAtK'],
            'SelectedCoverageAt10': selected['CatalogCoverageAtK'],
            'DiscoveryTuningLearners': len(tuning_ids),
            'ValidationLearnersUntouchedByWeightSelection': int(
                tables['learners']['cohort'].eq('validation').sum()
            ),
        })
    del tables, component_rows, popularity_rows
    gc.collect()

hybrid_tuning_metrics = pd.DataFrame(tuning_metric_rows)
hybrid_weight_selection = pd.DataFrame(selection_rows)
hybrid_tuning_metrics.to_csv(HYBRID_TUNING_METRICS_PATH, index=False)
hybrid_weight_selection.to_csv(HYBRID_WEIGHT_SELECTION_PATH, index=False)
assert not hybrid_weight_selection.duplicated(['Dataset', 'Task', 'Variant']).any()
assert hybrid_weight_selection['ValidationLearnersUntouchedByWeightSelection'].gt(0).all()
assert hybrid_tuning_metrics['TuningSource'].eq('discovery_only').all()
display(hybrid_weight_selection.sort_values(['Dataset', 'Task', 'Role', 'Variant']))
print(f'Tested {len(hybrid_tuning_metrics):,} discovery-only hybrid weight configurations.')

,Dataset,Task,Variant,Role,CanSelectProductionModel,SelectedWeightSignature,SelectedWeightsJSON,SelectedNDCGAt10,SelectedRecallAt10,SelectedCoverageAt10,DiscoveryTuningLearners,ValidationLearnersUntouchedByWeightSelection
3,ASSISTments,problem,hybrid_with_cluster_popularity,cluster_ablation,True,learner_neighbor_cf=1.00|sequential_transition...,"{""cluster_popularity"": 0.0, ""learner_neighbor_...",0.498437,0.265667,0.194474,5334,6667
2,ASSISTments,problem,hybrid_with_same_cluster_neighbor,cluster_ablation,True,learner_neighbor_cf=1.00|sequential_transition...,"{""learner_neighbor_cf"": 1.0, ""same_cluster_nei...",0.498437,0.265667,0.194474,5334,6667
0,ASSISTments,problem,hybrid_neighbor_sequential,production_candidate,True,learner_neighbor_cf=1.00|sequential_transition...,"{""learner_neighbor_cf"": 1.0, ""sequential_trans...",0.498437,0.265667,0.194474,5334,6667
1,ASSISTments,problem,hybrid_problem_with_content,research_ablation,False,learner_neighbor_cf=1.00|sequential_transition...,"{""content_problem"": 0.0, ""learner_neighbor_cf""...",0.498437,0.265667,0.194474,5334,6667
7,ASSISTments,skill,hybrid_with_cluster_popularity,cluster_ablation,False,learner_neighbor_cf=1.00|sequential_transition...,"{""cluster_popularity"": 0.0, ""learner_neighbor_...",0.743423,0.694654,0.962733,5334,6667
6,ASSISTments,skill,hybrid_with_same_cluster_neighbor,cluster_ablation,False,learner_neighbor_cf=1.00|sequential_transition...,"{""learner_neighbor_cf"": 1.0, ""same_cluster_nei...",0.743423,0.694654,0.962733,5334,6667
4,ASSISTments,skill,hybrid_neighbor_sequential,production_candidate,False,learner_neighbor_cf=1.00|sequential_transition...,"{""learner_neighbor_cf"": 1.0, ""sequential_trans...",0.743423,0.694654,0.962733,5334,6667
5,ASSISTments,skill,hybrid_skill_with_weakness,research_ablation,False,learner_neighbor_cf=1.00|sequential_transition...,"{""learner_neighbor_cf"": 1.0, ""sequential_trans...",0.743423,0.694654,0.962733,5334,6667
11,KDD,problem,hybrid_with_cluster_popularity,cluster_ablation,False,learner_neighbor_cf=0.25|sequential_transition...,"{""cluster_popularity"": 0.0, ""learner_neighbor_...",0.706865,0.445117,0.419883,91,113
10,KDD,problem,hybrid_with_same_cluster_neighbor,cluster_ablation,False,learner_neighbor_cf=0.25|sequential_transition...,"{""learner_neighbor_cf"": 0.25, ""same_cluster_ne...",0.706865,0.445117,0.419883,91,113


Tested 200 discovery-only hybrid weight configurations.


## Step 8 — Apply selected weights to the all-discovery refit outputs

Phase 3 already refit every selected base component using all discovery learners and saved its validation recommendations. This step reuses those outputs, removes sequential popularity fallback from genuine sequential evidence, applies the discovery-selected weights and streams final hybrid recommendations.

In [12]:
PHASE3_RECOMMENDATION_PATH = BASELINE_ROOT / 'validation_recommendations.parquet'
HYBRID_RECOMMENDATION_PATH = OUTPUT_ROOT / 'step8_hybrid_recommendations.parquet'
phase3_recommendation_dataset = pds.dataset(PHASE3_RECOMMENDATION_PATH, format='parquet')
RANK_COLUMNS = [f'rank__{component}' for component in FUSION_COMPONENTS]
CONTRIBUTION_COLUMNS = [f'contribution__{component}' for component in FUSION_COMPONENTS]
HYBRID_RECOMMENDATION_COLUMNS = [
    'Dataset', 'Task', 'Variant', 'VariantRole', 'CanSelectProductionModel',
    'CandidatePolicy', 'RelevanceDefinition', 'learner_id', 'item_id',
    'score', 'rank', 'score_source', 'FallbackUsed', 'WeightSignature',
] + RANK_COLUMNS + CONTRIBUTION_COLUMNS
hybrid_recommendation_schema = pa.schema([
    pa.field('Dataset', pa.string()), pa.field('Task', pa.string()),
    pa.field('Variant', pa.string()), pa.field('VariantRole', pa.string()),
    pa.field('CanSelectProductionModel', pa.bool_()),
    pa.field('CandidatePolicy', pa.string()),
    pa.field('RelevanceDefinition', pa.string()),
    pa.field('learner_id', pa.string()), pa.field('item_id', pa.string()),
    pa.field('score', pa.float64()), pa.field('rank', pa.int32()),
    pa.field('score_source', pa.string()), pa.field('FallbackUsed', pa.bool_()),
    pa.field('WeightSignature', pa.string()),
] + [pa.field(column, pa.float64()) for column in RANK_COLUMNS + CONTRIBUTION_COLUMNS])

def load_phase3_component_rows(dataset, task, policy, relevance, models):
    expression = (
        (pds.field('Dataset') == dataset)
        & (pds.field('Task') == task)
        & (pds.field('CandidatePolicy') == policy)
        & (pds.field('RelevanceDefinition') == relevance)
        & pds.field('Model').isin(list(models))
    )
    columns = [
        'Dataset', 'Task', 'Model', 'learner_id', 'item_id',
        'score', 'rank', 'score_source',
    ]
    frame = phase3_recommendation_dataset.to_table(
        filter=expression, columns=columns,
    ).to_pandas()
    parts = []
    for component, group in frame.groupby('Model', sort=False):
        standard = standardise_component_recommendations(
            group[['learner_id', 'item_id', 'score', 'rank', 'score_source']],
            component, dataset, task, component,
        )
        parts.append(standard)
    if not parts:
        return pd.DataFrame(columns=[
            'Dataset', 'Task', 'Component', 'learner_id', 'item_id',
            'score', 'rank', 'score_source',
        ])
    return pd.concat(parts, ignore_index=True)

def normalise_hybrid_output(
    recommendations, dataset, task, variant, role, can_select, policy, relevance,
):
    frame = recommendations.copy()
    frame.insert(0, 'RelevanceDefinition', relevance)
    frame.insert(0, 'CandidatePolicy', policy)
    frame.insert(0, 'CanSelectProductionModel', bool(can_select))
    frame.insert(0, 'VariantRole', role)
    frame.insert(0, 'Variant', variant)
    frame.insert(0, 'Task', task)
    frame.insert(0, 'Dataset', dataset)
    frame['learner_id'] = frame['learner_id'].astype(str)
    frame['item_id'] = frame['item_id'].astype(str)
    frame['score'] = frame['score'].astype(float)
    frame['rank'] = frame['rank'].astype('int32')
    frame['FallbackUsed'] = frame['FallbackUsed'].astype(bool)
    for column in RANK_COLUMNS:
        frame[column] = pd.to_numeric(frame[column], errors='coerce').astype(float)
    for column in CONTRIBUTION_COLUMNS:
        frame[column] = pd.to_numeric(frame[column], errors='coerce').fillna(0.0).astype(float)
    return frame[HYBRID_RECOMMENDATION_COLUMNS]

hybrid_writer = pq.ParquetWriter(
    HYBRID_RECOMMENDATION_PATH, hybrid_recommendation_schema, compression='snappy',
)
hybrid_recommendation_rows = []
try:
    for task_definition in TASK_DEFINITIONS:
        dataset, task = task_definition['Dataset'], task_definition['Task']
        learners = pd.read_parquet(DATASET_DIRS[dataset] / 'learner_splits.parquet')
        validation_ids = sorted(learners.loc[
            learners['cohort'].eq('validation'), 'learner_id'
        ].astype(str))
        task_variants = hybrid_variant_table[
            hybrid_variant_table['Dataset'].eq(dataset)
            & hybrid_variant_table['Task'].eq(task)
        ]
        required_models = (
            set(task_variants['Components'].map(json.loads).explode())
            | {FALLBACK_COMPONENT}
        )
        for relevance in RELEVANCE_DEFINITIONS:
            for policy in CANDIDATE_POLICIES:
                component_rows = load_phase3_component_rows(
                    dataset, task, policy, relevance, required_models,
                )
                popularity_rows = component_rows[
                    component_rows['Component'].eq(FALLBACK_COMPONENT)
                ].copy()
                assert set(popularity_rows['learner_id'].astype(str)).issubset(set(validation_ids))
                for variant_row in task_variants.itertuples(index=False):
                    selection = hybrid_weight_selection[
                        hybrid_weight_selection['Dataset'].eq(dataset)
                        & hybrid_weight_selection['Task'].eq(task)
                        & hybrid_weight_selection['Variant'].eq(variant_row.Variant)
                    ].iloc[0]
                    components = tuple(json.loads(variant_row.Components))
                    weights = json.loads(selection['SelectedWeightsJSON'])
                    recommendations = fuse_ranked_components(
                        component_rows, components, weights, validation_ids,
                        popularity_rows,
                    )
                    output = normalise_hybrid_output(
                        recommendations, dataset, task, variant_row.Variant,
                        variant_row.Role, variant_row.CanSelectProductionModel,
                        policy, relevance,
                    )
                    hybrid_writer.write_table(pa.Table.from_pandas(
                        output, schema=hybrid_recommendation_schema,
                        preserve_index=False,
                    ))
                    hybrid_recommendation_rows.append({
                        'Dataset': dataset, 'Task': task,
                        'Variant': variant_row.Variant,
                        'CandidatePolicy': policy,
                        'RelevanceDefinition': relevance,
                        'Rows': len(output),
                        'Recipients': output['learner_id'].nunique(),
                        'FallbackRecipients': output.loc[
                            output['FallbackUsed'], 'learner_id'
                        ].nunique(),
                    })
                    del recommendations, output
                del component_rows, popularity_rows
                gc.collect()
finally:
    hybrid_writer.close()

hybrid_recommendation_diagnostics = pd.DataFrame(hybrid_recommendation_rows)
assert (
    pq.ParquetFile(HYBRID_RECOMMENDATION_PATH).metadata.num_rows
    == hybrid_recommendation_diagnostics['Rows'].sum()
)
assert hybrid_recommendation_diagnostics['Rows'].gt(0).all()
display(hybrid_recommendation_diagnostics)

,Dataset,Task,Variant,CandidatePolicy,RelevanceDefinition,Rows,Recipients,FallbackRecipients
0,ASSISTments,problem,hybrid_neighbor_sequential,all_supported,attempted,126292,6667,290
1,ASSISTments,problem,hybrid_problem_with_content,all_supported,attempted,126292,6667,290
2,ASSISTments,problem,hybrid_with_same_cluster_neighbor,all_supported,attempted,126292,6667,290
3,ASSISTments,problem,hybrid_with_cluster_popularity,all_supported,attempted,126292,6667,290
4,ASSISTments,problem,hybrid_neighbor_sequential,novel_only,attempted,123926,6667,390
...,...,...,...,...,...,...,...,...
59,KDD,skill,hybrid_with_cluster_popularity,all_supported,successful,2257,113,0
60,KDD,skill,hybrid_neighbor_sequential,novel_only,successful,2007,113,0
61,KDD,skill,hybrid_skill_with_weakness,novel_only,successful,2007,113,0
62,KDD,skill,hybrid_with_same_cluster_neighbor,novel_only,successful,2007,113,0


## Step 9 — Verify explicit cold-start and missing-evidence routing

A learner receives discovery-early popularity only when the selected weighted components produce no positive reciprocal-rank candidate. Fallback rows have zero component contribution, personalized rows must have positive documented evidence. Diagnostics separate all, cold-start and non-cold-start learners.

In [13]:
COLD_START_DIAGNOSTICS_PATH = OUTPUT_ROOT / 'hybrid_cold_start_diagnostics.csv'
hybrid_dataset = pds.dataset(HYBRID_RECOMMENDATION_PATH, format='parquet')
cold_start_rows = []
for task_definition in TASK_DEFINITIONS:
    dataset, task = task_definition['Dataset'], task_definition['Task']
    learners = pd.read_parquet(DATASET_DIRS[dataset] / 'learner_splits.parquet')
    learners['learner_id'] = learners['learner_id'].astype(str)
    validation = learners[learners['cohort'].eq('validation')][
        ['learner_id', task_definition['ColdStartColumn']]
    ].copy()
    task_variants = hybrid_variant_table[
        hybrid_variant_table['Dataset'].eq(dataset)
        & hybrid_variant_table['Task'].eq(task)
    ]
    for relevance in RELEVANCE_DEFINITIONS:
        for policy in CANDIDATE_POLICIES:
            for variant in task_variants['Variant']:
                expression = (
                    (pds.field('Dataset') == dataset)
                    & (pds.field('Task') == task)
                    & (pds.field('Variant') == variant)
                    & (pds.field('CandidatePolicy') == policy)
                    & (pds.field('RelevanceDefinition') == relevance)
                )
                frame = hybrid_dataset.to_table(
                    filter=expression,
                    columns=[
                        'learner_id', 'score_source', 'FallbackUsed',
                        *CONTRIBUTION_COLUMNS,
                    ],
                ).to_pandas()
                frame['learner_id'] = frame['learner_id'].astype(str)
                contribution_total = frame[CONTRIBUTION_COLUMNS].sum(axis=1)
                assert frame.loc[frame['FallbackUsed'], 'score_source'].eq(
                    'popularity_fallback'
                ).all()
                assert np.isclose(
                    contribution_total[frame['FallbackUsed']], 0.0
                ).all()
                assert contribution_total[~frame['FallbackUsed']].gt(0).all()
                assert frame.groupby('learner_id')['FallbackUsed'].nunique().le(1).all()
                per_user = frame.groupby('learner_id').agg(
                    RecommendationCount=('learner_id', 'size'),
                    FallbackUsed=('FallbackUsed', 'first'),
                ).reset_index()
                audit = validation.merge(per_user, on='learner_id', how='left')
                audit['RecommendationCount'] = audit['RecommendationCount'].fillna(0).astype(int)
                audit['FallbackUsed'] = audit['FallbackUsed'].fillna(False).astype(bool)
                segments = {
                    'all': pd.Series(True, index=audit.index),
                    'cold_start': audit[task_definition['ColdStartColumn']].astype(bool),
                    'non_cold_start': ~audit[task_definition['ColdStartColumn']].astype(bool),
                }
                for segment, mask in segments.items():
                    subset = audit[mask]
                    if subset.empty:
                        continue
                    cold_start_rows.append({
                        'Dataset': dataset, 'Task': task, 'Variant': variant,
                        'CandidatePolicy': policy,
                        'RelevanceDefinition': relevance,
                        'Segment': segment, 'Learners': len(subset),
                        'LearnersWithRecommendations': subset['RecommendationCount'].gt(0).sum(),
                        'FallbackLearners': subset['FallbackUsed'].sum(),
                        'NoRecommendationLearners': subset['RecommendationCount'].eq(0).sum(),
                        'FallbackRate': subset['FallbackUsed'].mean(),
                        'MeanRecommendations': subset['RecommendationCount'].mean(),
                        'MinimumRecommendations': subset['RecommendationCount'].min(),
                        'MaximumRecommendations': subset['RecommendationCount'].max(),
                    })
                del frame, audit, per_user

hybrid_cold_start_diagnostics = pd.DataFrame(cold_start_rows)
hybrid_cold_start_diagnostics.to_csv(COLD_START_DIAGNOSTICS_PATH, index=False)
assert hybrid_cold_start_diagnostics['MinimumRecommendations'].ge(0).all()
assert hybrid_cold_start_diagnostics['MaximumRecommendations'].le(MAX_RECOMMENDATIONS).all()
display(hybrid_cold_start_diagnostics.sort_values([
    'Dataset', 'Task', 'Variant', 'CandidatePolicy',
    'RelevanceDefinition', 'Segment',
]))

,Dataset,Task,Variant,CandidatePolicy,RelevanceDefinition,Segment,Learners,LearnersWithRecommendations,FallbackLearners,NoRecommendationLearners,FallbackRate,MeanRecommendations,MinimumRecommendations,MaximumRecommendations
0,ASSISTments,problem,hybrid_neighbor_sequential,all_supported,attempted,all,6667,6667,290,0,0.043498,18.942853,1,20
1,ASSISTments,problem,hybrid_neighbor_sequential,all_supported,attempted,cold_start,164,164,164,0,1.000000,20.000000,20,20
2,ASSISTments,problem,hybrid_neighbor_sequential,all_supported,attempted,non_cold_start,6503,6503,126,0,0.019376,18.916193,1,20
24,ASSISTments,problem,hybrid_neighbor_sequential,all_supported,successful,all,6667,6667,301,0,0.045148,18.814309,1,20
25,ASSISTments,problem,hybrid_neighbor_sequential,all_supported,successful,cold_start,164,164,164,0,1.000000,20.000000,20,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,KDD,skill,hybrid_with_same_cluster_neighbor,all_supported,successful,non_cold_start,113,113,0,0,0.000000,19.973451,18,20
140,KDD,skill,hybrid_with_same_cluster_neighbor,novel_only,attempted,all,113,113,0,0,0.000000,18.646018,3,20
141,KDD,skill,hybrid_with_same_cluster_neighbor,novel_only,attempted,non_cold_start,113,113,0,0,0.000000,18.646018,3,20
156,KDD,skill,hybrid_with_same_cluster_neighbor,novel_only,successful,all,113,113,0,0,0.000000,17.761062,2,20


## Step 10 — Ranking, coverage and comparator evaluation

Every hybrid is evaluated at K = 5, 10 and 20 for both relevance definitions, both candidate policies and all available learner segments. Novel-only relevance excludes seen relevant items. The primary point-comparison table places each hybrid beside ordinary neighbour CF, sequential transitions and discovery-early popularity on identical evaluation slices.

In [14]:
def evaluate_hybrid_recommendations(
    recommendations, relevance, catalog, learners, task_definition,
    candidate_policy, relevance_name, variant,
):
    relevance_column = RELEVANCE_DEFINITIONS[relevance_name]
    ranked = recommendations.copy()
    ranked['learner_id'] = ranked['learner_id'].astype(str)
    ranked['item_id'] = ranked['item_id'].astype(str)
    ranked = ranked.sort_values(['learner_id', 'rank', 'item_id'], kind='mergesort')
    assert not ranked.duplicated(['learner_id', 'item_id']).any()
    assert set(ranked['item_id']).issubset(set(catalog['item_id'].astype(str)))
    relevance = relevance.copy()
    relevance['learner_id'] = relevance['learner_id'].astype(str)
    relevance['item_id'] = relevance['item_id'].astype(str)
    relevance_mask = relevance['in_candidate_catalog'] & relevance[relevance_column].eq(1)
    if candidate_policy == 'novel_only':
        relevance_mask &= ~relevance['seen_in_early'].fillna(False).astype(bool)
    relevant = relevance.loc[
        relevance_mask, ['learner_id', 'item_id']
    ].drop_duplicates()
    relevant_by_user = relevant.groupby('learner_id')['item_id'].agg(set).to_dict()
    ranked_by_user = ranked.groupby('learner_id')['item_id'].agg(list).to_dict()
    learners = learners.copy()
    learners['learner_id'] = learners['learner_id'].astype(str)
    validation = learners[learners['cohort'].eq('validation')].copy()
    eligible = set(validation.loc[
        validation[task_definition['EvaluableColumn']], 'learner_id'
    ]) & set(relevant_by_user)
    segments = {
        'all': eligible,
        'cold_start': eligible & set(validation.loc[
            validation[task_definition['ColdStartColumn']], 'learner_id'
        ]),
        'non_cold_start': eligible & set(validation.loc[
            ~validation[task_definition['ColdStartColumn']], 'learner_id'
        ]),
    }
    rows = []
    for segment, users in segments.items():
        if not users:
            continue
        for k in METRIC_KS:
            user_metrics = []
            union = set()
            lengths = []
            for learner_id in sorted(users):
                items = ranked_by_user.get(learner_id, [])[:k]
                user_metrics.append(ranking_metrics_for_user(
                    items, relevant_by_user[learner_id], k,
                ))
                union.update(items)
                lengths.append(len(items))
            means = pd.DataFrame(user_metrics).mean().to_dict()
            rows.append({
                'Dataset': task_definition['Dataset'],
                'Task': task_definition['Task'], 'Model': variant,
                'CandidatePolicy': candidate_policy,
                'RelevanceDefinition': relevance_name,
                'Segment': segment, 'K': k, **means,
                'CatalogCoverageAtK': len(union) / len(catalog),
                'MeanRecommendations': float(np.mean(lengths)),
                'EvaluatedLearners': len(users),
                'EvaluableRate': len(eligible) / len(validation),
                'EvaluationScope': 'frozen_validation_post_selection',
            })
    return pd.DataFrame(rows)

print('Defined corrected hybrid ranking and coverage evaluation.')

Defined corrected hybrid ranking and coverage evaluation.


In [15]:
HYBRID_METRICS_PATH = OUTPUT_ROOT / 'hybrid_metrics.csv'
HYBRID_COVERAGE_PATH = OUTPUT_ROOT / 'hybrid_coverage.csv'
STEP10_ALL_COMPARATORS_PATH = OUTPUT_ROOT / 'step10_metrics_with_comparators.csv'
STEP10_COMPARISON_PATH = OUTPUT_ROOT / 'step10_model_comparison.csv'
STEPS_5_10_CONFIG_PATH = OUTPUT_ROOT / 'steps_5_10_config.json'
hybrid_metric_parts = []
for task_definition in TASK_DEFINITIONS:
    dataset, task = task_definition['Dataset'], task_definition['Task']
    directory = DATASET_DIRS[dataset]
    learners = pd.read_parquet(directory / 'learner_splits.parquet')
    catalog = pd.read_parquet(directory / task_definition['CatalogFile'])
    relevance = pd.read_parquet(directory / task_definition['FutureRelevanceFile'])
    catalog['item_id'] = catalog['item_id'].astype(str)
    task_variants = hybrid_variant_table[
        hybrid_variant_table['Dataset'].eq(dataset)
        & hybrid_variant_table['Task'].eq(task)
    ]
    for relevance_name in RELEVANCE_DEFINITIONS:
        for policy in CANDIDATE_POLICIES:
            for variant_row in task_variants.itertuples(index=False):
                expression = (
                    (pds.field('Dataset') == dataset)
                    & (pds.field('Task') == task)
                    & (pds.field('Variant') == variant_row.Variant)
                    & (pds.field('CandidatePolicy') == policy)
                    & (pds.field('RelevanceDefinition') == relevance_name)
                )
                recommendations = hybrid_dataset.to_table(
                    filter=expression,
                    columns=['learner_id', 'item_id', 'score', 'rank'],
                ).to_pandas()
                metrics = evaluate_hybrid_recommendations(
                    recommendations, relevance, catalog, learners,
                    task_definition, policy, relevance_name, variant_row.Variant,
                )
                metrics.insert(3, 'VariantRole', variant_row.Role)
                metrics.insert(4, 'CanSelectProductionModel', bool(
                    variant_row.CanSelectProductionModel
                ))
                hybrid_metric_parts.append(metrics)
                del recommendations, metrics
    del learners, catalog, relevance
    gc.collect()

hybrid_metrics = pd.concat(hybrid_metric_parts, ignore_index=True)
hybrid_coverage = hybrid_metrics[[
    'Dataset', 'Task', 'Model', 'VariantRole', 'CanSelectProductionModel',
    'CandidatePolicy', 'RelevanceDefinition', 'Segment', 'K',
    'CatalogCoverageAtK', 'MeanRecommendations', 'EvaluatedLearners',
    'EvaluableRate',
]].copy()
hybrid_metrics.to_csv(HYBRID_METRICS_PATH, index=False)
hybrid_coverage.to_csv(HYBRID_COVERAGE_PATH, index=False)

phase3_metrics = pd.read_csv(BASELINE_ROOT / 'recommendation_metrics.csv')
COMPARATOR_MODELS = [
    'learner_neighbor_cf', 'sequential_transition',
    'popularity_discovery_early',
]
comparator_metrics = phase3_metrics[
    phase3_metrics['Model'].isin(COMPARATOR_MODELS)
].copy()
hybrid_metrics_with_source = hybrid_metrics.copy()
hybrid_metrics_with_source['ResultSource'] = 'phase4_hybrid'
comparator_metrics_with_source = comparator_metrics.copy()
comparator_metrics_with_source['ResultSource'] = 'verified_phase3_comparator'
metrics_with_comparators = pd.concat(
    [hybrid_metrics_with_source, comparator_metrics_with_source],
    ignore_index=True, sort=False,
)
metrics_with_comparators.to_csv(STEP10_ALL_COMPARATORS_PATH, index=False)
comparison_keys = [
    'Dataset', 'Task', 'CandidatePolicy', 'RelevanceDefinition', 'Segment', 'K',
]
cf_counts = comparator_metrics[
    comparator_metrics['Model'].eq('learner_neighbor_cf')
][comparison_keys + ['EvaluatedLearners']].rename(
    columns={'EvaluatedLearners': 'ComparatorEvaluatedLearners'}
)
count_check = hybrid_metrics.merge(
    cf_counts, on=comparison_keys, how='left', validate='many_to_one',
)
assert count_check['ComparatorEvaluatedLearners'].notna().all()
assert count_check['EvaluatedLearners'].eq(
    count_check['ComparatorEvaluatedLearners']
).all()

primary_slice = (
    hybrid_metrics['CandidatePolicy'].eq(PRIMARY_CANDIDATE_POLICY)
    & hybrid_metrics['RelevanceDefinition'].eq(PRIMARY_RELEVANCE)
    & hybrid_metrics['Segment'].eq(PRIMARY_SEGMENT)
    & hybrid_metrics['K'].eq(PRIMARY_K)
)
hybrid_point_comparison = hybrid_metrics[primary_slice].copy()
comparator_slice = comparator_metrics[
    comparator_metrics['CandidatePolicy'].eq(PRIMARY_CANDIDATE_POLICY)
    & comparator_metrics['RelevanceDefinition'].eq(PRIMARY_RELEVANCE)
    & comparator_metrics['Segment'].eq(PRIMARY_SEGMENT)
    & comparator_metrics['K'].eq(PRIMARY_K)
]
for comparator in COMPARATOR_MODELS:
    label = {
        'learner_neighbor_cf': 'NeighborCF',
        'sequential_transition': 'Sequential',
        'popularity_discovery_early': 'Popularity',
    }[comparator]
    values = comparator_slice[comparator_slice['Model'].eq(comparator)][[
        'Dataset', 'Task', 'RecallAtK', 'NDCGAtK', 'CatalogCoverageAtK',
    ]].rename(columns={
        'RecallAtK': f'{label}RecallAtK',
        'NDCGAtK': f'{label}NDCGAtK',
        'CatalogCoverageAtK': f'{label}CatalogCoverageAtK',
    })
    hybrid_point_comparison = hybrid_point_comparison.merge(
        values, on=['Dataset', 'Task'], how='left', validate='many_to_one',
    )
hybrid_point_comparison['RecallDeltaVsNeighborCF'] = (
    hybrid_point_comparison['RecallAtK']
    - hybrid_point_comparison['NeighborCFRecallAtK']
)
hybrid_point_comparison['NDCGDeltaVsNeighborCF'] = (
    hybrid_point_comparison['NDCGAtK']
    - hybrid_point_comparison['NeighborCFNDCGAtK']
)
hybrid_point_comparison['CoverageRatioVsNeighborCF'] = (
    hybrid_point_comparison['CatalogCoverageAtK']
    / hybrid_point_comparison['NeighborCFCatalogCoverageAtK'].replace(0, np.nan)
).fillna(0.0)
hybrid_point_comparison.to_csv(STEP10_COMPARISON_PATH, index=False)

steps_5_10_config = {
    'phase': 4, 'implemented_steps': list(range(5, 11)),
    'fusion': 'weighted_reciprocal_rank', 'rrf_constant': RRF_CONSTANT,
    'component_top_k': MAX_RECOMMENDATIONS,
    'weight_grid_step': WEIGHT_GRID_STEP,
    'weight_selection_source': 'discovery_only',
    'weight_selection_slice': {
        'relevance': PRIMARY_RELEVANCE,
        'candidate_policy': PRIMARY_CANDIDATE_POLICY,
        'segment': PRIMARY_SEGMENT, 'k': PRIMARY_K, 'metric': 'NDCGAtK',
    },
    'successful_sensitivity_uses_attempted_weights': True,
    'sequential_evidence_filter': "score_source == 'transition'",
    'fallback': FALLBACK_COMPONENT,
    'fallback_condition': 'no_positive_weight_personalised_candidate',
    'validation_scoring_source': 'verified_phase3_all_discovery_refit_recommendations',
    'validation_inference_scope': 'frozen_cohort_post_selection_not_independent_test',
    'novel_relevance_excludes_seen': True,
    'dense_learner_item_matrix_constructed': False,
}
with open(STEPS_5_10_CONFIG_PATH, 'w', encoding='utf-8') as file:
    json.dump(steps_5_10_config, file, indent=2)

steps_5_10_paths = {
    'tuning_components': TUNING_COMPONENT_PATH,
    'discovery_split': TUNING_SPLIT_PATH,
    'tuning_metrics': HYBRID_TUNING_METRICS_PATH,
    'weight_selection': HYBRID_WEIGHT_SELECTION_PATH,
    'recommendations': HYBRID_RECOMMENDATION_PATH,
    'cold_start_diagnostics': COLD_START_DIAGNOSTICS_PATH,
    'metrics': HYBRID_METRICS_PATH,
    'coverage': HYBRID_COVERAGE_PATH,
    'all_metrics_with_comparators': STEP10_ALL_COMPARATORS_PATH,
    'point_comparison': STEP10_COMPARISON_PATH,
    'config': STEPS_5_10_CONFIG_PATH,
}
steps_5_10_manifest = pd.DataFrame([
    {
        'Artifact': name, 'File': path.name,
        'Rows': (
            pq.ParquetFile(path).metadata.num_rows
            if path.suffix == '.parquet'
            else (len(pd.read_csv(path)) if path.suffix == '.csv' else 1)
        ),
        'Bytes': path.stat().st_size,
    }
    for name, path in steps_5_10_paths.items()
])
steps_5_10_manifest.to_csv(
    OUTPUT_ROOT / 'steps_5_10_artifact_manifest.csv', index=False,
)
assert len(steps_5_10_manifest) == len(steps_5_10_paths)
display(hybrid_point_comparison.sort_values([
    'Dataset', 'Task', 'NDCGAtK'
], ascending=[True, True, False]))
display(steps_5_10_manifest)

,Dataset,Task,Model,VariantRole,CanSelectProductionModel,CandidatePolicy,RelevanceDefinition,Segment,K,PrecisionAtK,...,NeighborCFCatalogCoverageAtK,SequentialRecallAtK,SequentialNDCGAtK,SequentialCatalogCoverageAtK,PopularityRecallAtK,PopularityNDCGAtK,PopularityCatalogCoverageAtK,RecallDeltaVsNeighborCF,NDCGDeltaVsNeighborCF,CoverageRatioVsNeighborCF
0,ASSISTments,problem,hybrid_neighbor_sequential,production_candidate,True,all_supported,attempted,all,10,0.439626,...,0.219513,0.116604,0.219903,0.381005,0.007234,0.011900,0.000251,5.551115e-17,0.000000,1.000000
1,ASSISTments,problem,hybrid_problem_with_content,research_ablation,False,all_supported,attempted,all,10,0.439626,...,0.219513,0.116604,0.219903,0.381005,0.007234,0.011900,0.000251,5.551115e-17,0.000000,1.000000
2,ASSISTments,problem,hybrid_with_same_cluster_neighbor,cluster_ablation,True,all_supported,attempted,all,10,0.439626,...,0.219513,0.116604,0.219903,0.381005,0.007234,0.011900,0.000251,5.551115e-17,0.000000,1.000000
3,ASSISTments,problem,hybrid_with_cluster_popularity,cluster_ablation,True,all_supported,attempted,all,10,0.439626,...,0.219513,0.116604,0.219903,0.381005,0.007234,0.011900,0.000251,5.551115e-17,0.000000,1.000000
4,ASSISTments,skill,hybrid_neighbor_sequential,production_candidate,False,all_supported,attempted,all,10,0.467185,...,0.975155,0.432834,0.489179,0.975155,0.260358,0.255585,0.062112,9.135685e-03,0.006613,1.000000
5,ASSISTments,skill,hybrid_skill_with_weakness,research_ablation,False,all_supported,attempted,all,10,0.467185,...,0.975155,0.432834,0.489179,0.975155,0.260358,0.255585,0.062112,9.135685e-03,0.006613,1.000000
6,ASSISTments,skill,hybrid_with_same_cluster_neighbor,cluster_ablation,False,all_supported,attempted,all,10,0.467185,...,0.975155,0.432834,0.489179,0.975155,0.260358,0.255585,0.062112,9.135685e-03,0.006613,1.000000
7,ASSISTments,skill,hybrid_with_cluster_popularity,cluster_ablation,False,all_supported,attempted,all,10,0.467185,...,0.975155,0.432834,0.489179,0.975155,0.260358,0.255585,0.062112,9.135685e-03,0.006613,1.000000
8,KDD,problem,hybrid_neighbor_sequential,production_candidate,False,all_supported,attempted,all,10,0.518018,...,0.295906,0.370854,0.614744,0.443275,0.040382,0.059862,0.011696,1.261944e-01,0.159175,1.640316
9,KDD,problem,hybrid_problem_with_content,research_ablation,False,all_supported,attempted,all,10,0.518018,...,0.295906,0.370854,0.614744,0.443275,0.040382,0.059862,0.011696,1.261944e-01,0.159175,1.640316


,Artifact,File,Rows,Bytes
0,tuning_components,step6_tuning_component_recommendations.parquet,967302,4593266
1,discovery_split,step6_discovery_split.csv,54240,1872556
2,tuning_metrics,hybrid_tuning_metrics.csv,200,84220
3,weight_selection,hybrid_weight_selection.csv,16,4984
4,recommendations,step8_hybrid_recommendations.parquet,3782056,9555952
5,cold_start_diagnostics,hybrid_cold_start_diagnostics.csv,160,20051
6,metrics,hybrid_metrics.csv,480,132962
7,coverage,hybrid_coverage.csv,480,76233
8,all_metrics_with_comparators,step10_metrics_with_comparators.csv,840,256369
9,point_comparison,step10_model_comparison.csv,16,8450


# Part B — Confidence-Gated Hybrid Experiment

This section runs the gated experiment after the complete global-fusion experiment above.


## Step 1 — Isolated paths, experiment contract and input validation

In [16]:
BENCHMARK_ROOT = Path('/content/drive/MyDrive/datasets/recommendation_benchmark_final_outputs')
BASELINE_ROOT = Path('/content/drive/MyDrive/datasets/recommendation_baseline_outputs')
GLOBAL_HYBRID_ROOT = Path('/content/drive/MyDrive/datasets/hybrid_recommender_outputs')
GATED_OUTPUT_ROOT = Path(
    '/content/drive/MyDrive/datasets/hybrid_recommender_gated_outputs'
)
GATED_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ASSIST_ROOT = BENCHMARK_ROOT / 'assistments'

RANDOM_STATE = 42
DISCOVERY_TUNING_FRACTION = 0.20
CANDIDATE_DEPTH = 100
FINAL_K = 20
METRIC_KS = (5, 10, 20)
RRF_CONSTANT = 60.0
PRIMARY_K = 10
COVERAGE_FLOOR_RATIO = 0.50
NEIGHBOR_BATCH_SIZE = 512

REQUIRED_PHASE2 = [
    'learner_splits.parquet', 'problem_catalog.parquet',
    'early_problem_history.parquet', 'future_problem_relevance.parquet',
    'early_skill_history.parquet', 'problem_skill_map.parquet',
]
REQUIRED_PHASE3 = [
    'neighbor_configuration.csv', 'hyperparameter_selection.csv',
    'transition_probabilities.parquet', 'recommendation_metrics.csv',
    'step11_discovery_recent_items.parquet',
    'step11_validation_recent_items.parquet',
    'steps_11_12_artifact_manifest.csv', 'leakage_audit.csv',
]
for filename in REQUIRED_PHASE2:
    assert (ASSIST_ROOT / filename).is_file(), filename
for filename in REQUIRED_PHASE3:
    assert (BASELINE_ROOT / filename).is_file(), filename
assert GLOBAL_HYBRID_ROOT != GATED_OUTPUT_ROOT

leakage = pd.read_csv(BASELINE_ROOT / 'leakage_audit.csv')
flags = [
    'DiscoveryValidationDisjoint', 'RecommendationsValidationOnly',
    'CandidatesInFrozenCatalog', 'NovelRecommendationsExcludeSeen',
    'CatalogSupportsMaximumK',
]
assert leakage[flags].apply(lambda column: column.astype(str).str.lower().eq('true')).all().all()
sequence_manifest = pd.read_csv(
    BASELINE_ROOT / 'steps_11_12_artifact_manifest.csv'
).set_index('Artifact')
for artifact in ['discovery_recent_items', 'validation_recent_items']:
    row = sequence_manifest.loc[artifact]
    path = BASELINE_ROOT / row['File']
    assert path.stat().st_size == int(row['Bytes'])
    assert pq.ParquetFile(path).metadata.num_rows == int(row['Rows'])

neighbor_config = pd.read_csv(BASELINE_ROOT / 'neighbor_configuration.csv')
selection = pd.read_csv(BASELINE_ROOT / 'hyperparameter_selection.csv')
neighbor_row = neighbor_config[
    neighbor_config['Dataset'].eq('ASSISTments')
    & neighbor_config['Task'].eq('problem')
].iloc[0]
selection_row = selection[
    selection['Dataset'].eq('ASSISTments')
    & selection['Task'].eq('problem')
].iloc[0]
SELECTED_NEIGHBORS = int(neighbor_row['SelectedNeighbors'])
SELECTED_DECAY = float(selection_row['SelectedRecencyDecay'])
SELECTED_SHRINKAGE = float(selection_row['SelectedTransitionShrinkage'])
SELECTED_CONTENT_TOLERANCE = float(selection_row['SelectedDifficultyTolerance'])
assert SELECTED_NEIGHBORS == 20
assert neighbor_row['SelectedNeighborWeighting'] == 'cosine'

## Step 2 — Load the primary temporal benchmark and freeze discovery splits

In [17]:
learners = pd.read_parquet(ASSIST_ROOT / 'learner_splits.parquet')
catalog = pd.read_parquet(ASSIST_ROOT / 'problem_catalog.parquet')
early_problem = pd.read_parquet(ASSIST_ROOT / 'early_problem_history.parquet')
future_problem = pd.read_parquet(ASSIST_ROOT / 'future_problem_relevance.parquet')
for frame in [learners, early_problem, future_problem]:
    frame['learner_id'] = frame['learner_id'].astype(str)
for frame in [catalog, early_problem, future_problem]:
    frame['item_id'] = frame['item_id'].astype(str)

discovery_ids = sorted(learners.loc[learners['cohort'].eq('discovery'), 'learner_id'])
validation_ids = sorted(learners.loc[learners['cohort'].eq('validation'), 'learner_id'])
discovery_train_ids, discovery_tuning_ids = train_test_split(
    discovery_ids, test_size=DISCOVERY_TUNING_FRACTION, random_state=RANDOM_STATE,
)
discovery_train_ids = sorted(discovery_train_ids)
discovery_tuning_ids = sorted(discovery_tuning_ids)
assert set(discovery_train_ids).isdisjoint(discovery_tuning_ids)
assert set(discovery_ids).isdisjoint(validation_ids)
item_order = sorted(catalog['item_id'].unique())
split_table = pd.DataFrame({
    'learner_id': discovery_train_ids + discovery_tuning_ids + validation_ids,
    'Role': (['discovery_training'] * len(discovery_train_ids)
             + ['discovery_tuning'] * len(discovery_tuning_ids)
             + ['validation'] * len(validation_ids)),
})
split_table.to_csv(
    GATED_OUTPUT_ROOT / 'gated_experiment_split.csv', index=False
)
display(split_table.groupby('Role').size().rename('Learners').reset_index())

,Role,Learners
0,discovery_training,21334
1,discovery_tuning,5334
2,validation,6667


## Step 3 — Sparse top-100 component builders with confidence evidence

In [18]:
def sparse_history_matrix(history, learner_order, item_order):
    learner_index = {value: index for index, value in enumerate(learner_order)}
    item_index = {value: index for index, value in enumerate(item_order)}
    subset = history[
        history['learner_id'].isin(learner_index)
        & history['item_id'].isin(item_index)
        & history['in_candidate_catalog']
    ][['learner_id', 'item_id', 'early_interaction_count']].copy()
    matrix = csr_matrix((
        np.log1p(subset['early_interaction_count'].astype(float)),
        (subset['learner_id'].map(learner_index), subset['item_id'].map(item_index)),
    ), shape=(len(learner_order), len(item_order)), dtype=np.float32)
    matrix.eliminate_zeros()
    return matrix

def target_matrix(relevance, learner_order, item_order, relevance_column):
    learner_index = {value: index for index, value in enumerate(learner_order)}
    item_index = {value: index for index, value in enumerate(item_order)}
    labels = relevance[
        relevance['learner_id'].isin(learner_index)
        & relevance['item_id'].isin(item_index)
        & relevance['in_candidate_catalog']
        & relevance[relevance_column].eq(1)
    ][['learner_id', 'item_id']].drop_duplicates()
    return csr_matrix((
        np.ones(len(labels), dtype=np.float32),
        (labels['learner_id'].map(learner_index), labels['item_id'].map(item_index)),
    ), shape=(len(learner_order), len(item_order)), dtype=np.float32)

def neighbor_candidates(training_ids, query_ids, relevance_column):
    training_history = sparse_history_matrix(early_problem, training_ids, item_order)
    query_history = sparse_history_matrix(early_problem, query_ids, item_order)
    model = NearestNeighbors(
        n_neighbors=min(SELECTED_NEIGHBORS, len(training_ids)),
        metric='cosine', algorithm='brute', n_jobs=-1,
    ).fit(training_history)
    distances, indices = model.kneighbors(query_history)
    similarities = np.clip(1.0 - distances, 0.0, 1.0).astype(np.float32)
    targets = target_matrix(future_problem, training_ids, item_order, relevance_column)
    rows = []
    for start in range(0, len(query_ids), NEIGHBOR_BATCH_SIZE):
        end = min(start + NEIGHBOR_BATCH_SIZE, len(query_ids))
        local = np.repeat(np.arange(end - start), indices[start:end].shape[1])
        weights = csr_matrix((
            similarities[start:end].ravel(),
            (local, indices[start:end].ravel()),
        ), shape=(end - start, len(training_ids)), dtype=np.float32)
        totals = np.asarray(weights.sum(axis=1)).ravel()
        inverse = np.divide(1.0, totals, out=np.zeros_like(totals), where=totals > 0)
        scores = (diags(inverse) @ weights @ targets).tocsr()
        for local_index, learner_id in enumerate(query_ids[start:end]):
            left, right = scores.indptr[local_index], scores.indptr[local_index + 1]
            candidates = sorted([
                (float(value), item_order[item_index])
                for item_index, value in zip(scores.indices[left:right], scores.data[left:right])
                if value > 0
            ], key=lambda value: (-value[0], value[1]))[:CANDIDATE_DEPTH]
            cf_confidence = float(similarities[start + local_index].max())
            positive_neighbors = int((similarities[start + local_index] > 0).sum())
            for rank, (score, item_id) in enumerate(candidates, start=1):
                rows.append((
                    learner_id, item_id, score, rank,
                    cf_confidence, positive_neighbors,
                ))
    del training_history, query_history, model, distances, indices, similarities, targets
    gc.collect()
    return pd.DataFrame(rows, columns=[
        'learner_id', 'item_id', 'cf_score', 'cf_rank',
        'cf_confidence', 'positive_neighbors',
    ])

def sequential_candidates(query_ids, recent_path):
    recent = pd.read_parquet(
        recent_path, filters=[('Dataset', '==', 'ASSISTments'), ('Task', '==', 'problem')],
    )
    recent['learner_id'] = recent['learner_id'].astype(str)
    recent['source_item_id'] = recent['source_item_id'].astype(str)
    recent = recent[recent['learner_id'].isin(set(query_ids))].copy()
    transitions = pd.read_parquet(
        BASELINE_ROOT / 'transition_probabilities.parquet',
        filters=[('Dataset', '==', 'ASSISTments'), ('Task', '==', 'problem')],
    )
    transitions['source_item_id'] = transitions['source_item_id'].astype(str)
    transitions['target_item_id'] = transitions['target_item_id'].astype(str)
    transitions['probability'] = transitions['transition_count'] / (
        transitions['source_transition_total'] + SELECTED_SHRINKAGE
    )
    joined = recent.merge(
        transitions[['source_item_id', 'target_item_id', 'probability', 'transition_count']],
        on='source_item_id', how='inner', validate='many_to_many',
    )
    joined['recency_weight'] = np.power(SELECTED_DECAY, joined['recency_rank'].astype(float))
    joined['weighted_probability'] = joined['recency_weight'] * joined['probability']
    joined['weighted_confidence'] = joined['recency_weight'] * (
        joined['transition_count'] / (joined['transition_count'] + 10.0)
    )
    scored = joined.groupby(['learner_id', 'target_item_id'], sort=False).agg(
        sequential_score=('weighted_probability', 'sum'),
        sequential_confidence=('weighted_confidence', 'max'),
        transition_support=('transition_count', 'max'),
        supporting_sources=('source_item_id', 'nunique'),
    ).reset_index().rename(columns={'target_item_id': 'item_id'})
    scored = scored.sort_values(
        ['learner_id', 'sequential_score', 'item_id'],
        ascending=[True, False, True], kind='mergesort',
    ).groupby('learner_id', sort=False).head(CANDIDATE_DEPTH).copy()
    scored['sequential_rank'] = scored.groupby('learner_id', sort=False).cumcount() + 1
    return scored

def content_candidates(query_ids):
    skill_history = pd.read_parquet(ASSIST_ROOT / 'early_skill_history.parquet')
    problem_skill = pd.read_parquet(ASSIST_ROOT / 'problem_skill_map.parquet')
    skill_history['learner_id'] = skill_history['learner_id'].astype(str)
    skill_history['item_id'] = skill_history['item_id'].astype(str)
    problem_skill['problem_item_id'] = problem_skill['problem_item_id'].astype(str)
    problem_skill['skill_item_id'] = problem_skill['skill_item_id'].astype(str)
    needs = skill_history[
        skill_history['learner_id'].isin(set(query_ids))
        & skill_history['in_candidate_catalog']
        & skill_history['skill_state'].isin({'weak', 'developing'})
    ][['learner_id', 'item_id', 'empirical_bayes_mastery', 'mastery_evidence_confidence']].copy()
    needs['weakness'] = (
        (1.0 - needs['empirical_bayes_mastery'])
        * needs['mastery_evidence_confidence']
    )
    mapped = needs.merge(
        problem_skill[['problem_item_id', 'skill_item_id', 'association_share']],
        left_on='item_id', right_on='skill_item_id', how='inner',
    )
    mapped['alignment'] = mapped['weakness'] * mapped['association_share']
    mapped['mastery_part'] = mapped['empirical_bayes_mastery'] * mapped['association_share']
    scored = mapped.groupby(['learner_id', 'problem_item_id'], sort=False).agg(
        weak_alignment=('alignment', 'sum'),
        weighted_mastery=('mastery_part', 'sum'),
        association_coverage=('association_share', 'sum'),
    ).reset_index().rename(columns={'problem_item_id': 'item_id'})
    scored['profile_mastery'] = (
        scored['weighted_mastery'] / scored['association_coverage'].clip(lower=1e-12)
    ).clip(0, 1)
    metadata = catalog[['item_id', 'difficulty_proxy', 'training_popularity']].copy()
    metadata['difficulty_proxy'] = metadata['difficulty_proxy'].fillna(0.5).clip(0, 1)
    scored = scored.merge(metadata, on='item_id', how='inner')
    scored['difficulty_suitability'] = (
        1.0 - (scored['difficulty_proxy'] - scored['profile_mastery']).abs()
        / SELECTED_CONTENT_TOLERANCE
    ).clip(0, 1)
    scored['content_score'] = (
        0.75 * scored['weak_alignment'].clip(0, 1)
        + 0.25 * scored['difficulty_suitability']
        + 1e-9 * scored['training_popularity']
    )
    scored = scored.sort_values(
        ['learner_id', 'content_score', 'item_id'],
        ascending=[True, False, True], kind='mergesort',
    ).groupby('learner_id', sort=False).head(CANDIDATE_DEPTH).copy()
    scored['content_rank'] = scored.groupby('learner_id', sort=False).cumcount() + 1
    return scored[['learner_id', 'item_id', 'content_score', 'content_rank']]

def popularity_candidates(query_ids, candidate_policy):
    ranking = catalog[['item_id', 'training_interactions']].copy()
    ranking['popularity_score'] = ranking['training_interactions'].astype(float)
    ranking = ranking.sort_values(
        ['popularity_score', 'item_id'], ascending=[False, True], kind='mergesort'
    )
    seen_by_user = {}
    if candidate_policy == 'novel_only':
        seen = early_problem[
            early_problem['learner_id'].isin(set(query_ids))
            & early_problem['in_candidate_catalog']
        ][['learner_id', 'item_id']].drop_duplicates()
        seen_by_user = seen.groupby('learner_id')['item_id'].agg(set).to_dict()
    rows = []
    ordered = list(ranking[['item_id', 'popularity_score']].itertuples(index=False, name=None))
    for learner_id in query_ids:
        excluded = seen_by_user.get(learner_id, set())
        rank = 0
        for item_id, score in ordered:
            if item_id in excluded:
                continue
            rank += 1
            rows.append((learner_id, item_id, float(score), rank))
            if rank == FINAL_K:
                break
    return pd.DataFrame(rows, columns=['learner_id', 'item_id', 'popularity_score', 'rank'])


## Step 4 — Generate discovery-tuning candidates and measure complementarity

In [19]:
GATED_TUNING_COMPONENT_PATH = (
    GATED_OUTPUT_ROOT / 'gated_tuning_components.parquet'
)
tuning_cf = neighbor_candidates(
    discovery_train_ids, discovery_tuning_ids, 'relevance_binary',
)
tuning_sequence = sequential_candidates(
    discovery_tuning_ids, BASELINE_ROOT / 'step11_discovery_recent_items.parquet',
)
tuning_content = content_candidates(discovery_tuning_ids)
tuning_cf['Component'] = 'learner_neighbor_cf'
tuning_sequence['Component'] = 'sequential_transition'
tuning_content['Component'] = 'content_reranker'
pd.concat([
    tuning_cf.rename(columns={'cf_score': 'component_score', 'cf_rank': 'component_rank'})[[
        'Component', 'learner_id', 'item_id', 'component_score', 'component_rank'
    ]],
    tuning_sequence.rename(columns={
        'sequential_score': 'component_score', 'sequential_rank': 'component_rank'
    })[['Component', 'learner_id', 'item_id', 'component_score', 'component_rank']],
    tuning_content.rename(columns={
        'content_score': 'component_score', 'content_rank': 'component_rank'
    })[['Component', 'learner_id', 'item_id', 'component_score', 'component_rank']],
], ignore_index=True).to_parquet(
    GATED_TUNING_COMPONENT_PATH, index=False, compression='snappy'
)

tuning_relevant = future_problem[
    future_problem['learner_id'].isin(set(discovery_tuning_ids))
    & future_problem['in_candidate_catalog']
    & future_problem['relevance_binary'].eq(1)
][['learner_id', 'item_id']].drop_duplicates()
relevant_by_user = tuning_relevant.groupby('learner_id')['item_id'].agg(set).to_dict()
cf_top10 = (
    tuning_cf[tuning_cf['cf_rank'].le(10)]
    .groupby('learner_id')['item_id'].agg(set).to_dict()
)
seq_top10 = (
    tuning_sequence[tuning_sequence['sequential_rank'].le(10)]
    .groupby('learner_id')['item_id'].agg(set).to_dict()
)
complementarity_rows = []
for learner_id in sorted(set(relevant_by_user)):
    relevant = relevant_by_user[learner_id]
    cf_hits = cf_top10.get(learner_id, set()) & relevant
    seq_hits = seq_top10.get(learner_id, set()) & relevant
    complementarity_rows.append({
        'learner_id': learner_id,
        'RelevantItems': len(relevant), 'CFHitsAt10': len(cf_hits),
        'SequentialHitsAt10': len(seq_hits),
        'SharedHitsAt10': len(cf_hits & seq_hits),
        'CFOnlyHitsAt10': len(cf_hits - seq_hits),
        'SequentialOnlyHitsAt10': len(seq_hits - cf_hits),
        'SequentialAddsUniqueHit': bool(seq_hits - cf_hits),
    })
complementarity = pd.DataFrame(complementarity_rows)
complementarity.to_csv(
    GATED_OUTPUT_ROOT / 'component_complementarity.csv', index=False
)
complementarity_summary = pd.DataFrame([{
    'EvaluableTuningLearners': len(complementarity),
    'LearnersWithCFOnlyHits': complementarity['CFOnlyHitsAt10'].gt(0).sum(),
    'LearnersWithSequentialOnlyHits': complementarity['SequentialOnlyHitsAt10'].gt(0).sum(),
    'SequentialUniqueHitRate': complementarity['SequentialAddsUniqueHit'].mean(),
    'TotalCFOnlyHits': complementarity['CFOnlyHitsAt10'].sum(),
    'TotalSequentialOnlyHits': complementarity['SequentialOnlyHitsAt10'].sum(),
    'TotalSharedHits': complementarity['SharedHitsAt10'].sum(),
}])
display(complementarity_summary)

,EvaluableTuningLearners,LearnersWithCFOnlyHits,LearnersWithSequentialOnlyHits,SequentialUniqueHitRate,TotalCFOnlyHits,TotalSequentialOnlyHits,TotalSharedHits
0,4799,3119,2384,0.49677,19158,5382,2003


## Step 5 — Confidence-gated fusion and candidate-policy utilities

In [20]:
def filter_seen(frame, query_ids, candidate_policy):
    if candidate_policy == 'all_supported' or frame.empty:
        return frame.copy()
    if candidate_policy != 'novel_only':
        raise ValueError(candidate_policy)
    seen = early_problem[
        early_problem['learner_id'].isin(set(query_ids))
        & early_problem['in_candidate_catalog']
    ][['learner_id', 'item_id']].drop_duplicates().assign(_seen=True)
    merged = frame.merge(seen, on=['learner_id', 'item_id'], how='left')
    return merged[merged['_seen'].ne(True)].drop(columns='_seen')

def gated_fusion(
    cf, sequence, content, query_ids, candidate_policy,
    base_sequence_weight, minimum_transition_support, content_bonus,
):
    cf = filter_seen(cf, query_ids, candidate_policy)
    sequence = filter_seen(sequence, query_ids, candidate_policy)
    content = filter_seen(content, query_ids, candidate_policy)
    sequence = sequence[
        sequence['transition_support'].ge(minimum_transition_support)
    ].copy()
    candidates = cf.merge(
        sequence, on=['learner_id', 'item_id'], how='outer'
    ).merge(content, on=['learner_id', 'item_id'], how='outer')
    candidates = candidates[candidates['learner_id'].isin(set(query_ids))].copy()
    has_cf_by_user = candidates.groupby('learner_id')['cf_rank'].transform(
        lambda values: values.notna().any()
    )
    has_seq_by_user = candidates.groupby('learner_id')['sequential_rank'].transform(
        lambda values: values.notna().any()
    )
    learner_seq_confidence = candidates.groupby('learner_id')[
        'sequential_confidence'
    ].transform('max').fillna(0.0).clip(0, 1)
    candidates['effective_sequence_weight'] = (
        float(base_sequence_weight) * learner_seq_confidence
    )
    candidates.loc[~has_seq_by_user, 'effective_sequence_weight'] = 0.0
    candidates.loc[~has_cf_by_user & has_seq_by_user, 'effective_sequence_weight'] = 1.0
    candidates['effective_cf_weight'] = np.where(
        has_cf_by_user, 1.0 - candidates['effective_sequence_weight'], 0.0
    )
    candidates['cf_contribution'] = np.where(
        candidates['cf_rank'].notna(),
        candidates['effective_cf_weight'] / (RRF_CONSTANT + candidates['cf_rank']), 0.0,
    )
    candidates['sequential_contribution'] = np.where(
        candidates['sequential_rank'].notna(),
        candidates['effective_sequence_weight']
        / (RRF_CONSTANT + candidates['sequential_rank']),
        0.0,
    )
    candidates['content_contribution'] = np.where(
        candidates['content_rank'].notna(),
        float(content_bonus) / (RRF_CONSTANT + candidates['content_rank']), 0.0,
    )
    candidates['score'] = candidates[[
        'cf_contribution', 'sequential_contribution', 'content_contribution'
    ]].sum(axis=1)
    candidates = candidates[candidates['score'].gt(0)].copy()
    candidates = candidates.sort_values(
        ['learner_id', 'score', 'item_id'], ascending=[True, False, True], kind='mergesort'
    ).groupby('learner_id', sort=False).head(FINAL_K).copy()
    candidates['rank'] = candidates.groupby('learner_id', sort=False).cumcount() + 1
    candidates['FallbackUsed'] = False
    candidates['score_source'] = np.select([
        candidates['cf_contribution'].gt(0) & candidates['sequential_contribution'].gt(0),
        candidates['sequential_contribution'].gt(0),
        candidates['cf_contribution'].gt(0),
        candidates['content_contribution'].gt(0),
    ], ['cf+sequential', 'sequential', 'cf', 'content'], default='unknown')
    recipients = set(candidates['learner_id'])
    missing = sorted(set(query_ids) - recipients)
    fallback = popularity_candidates(missing, candidate_policy)
    if len(fallback):
        fallback['score'] = -fallback['rank'].astype(float)
        fallback['FallbackUsed'] = True
        fallback['score_source'] = 'popularity_fallback'
        for column in candidates.columns:
            if column not in fallback.columns:
                fallback[column] = 0.0 if column.endswith('contribution') else np.nan
        fallback = fallback[candidates.columns]
        candidates = pd.concat([candidates, fallback], ignore_index=True)
    candidates['BaseSequenceWeight'] = float(base_sequence_weight)
    candidates['MinimumTransitionSupport'] = int(minimum_transition_support)
    candidates['ContentBonus'] = float(content_bonus)
    assert not candidates.duplicated(['learner_id', 'item_id']).any()
    assert candidates.groupby('learner_id').size().le(FINAL_K).all()
    return candidates.sort_values(['learner_id', 'rank', 'item_id'], kind='mergesort')

def user_metrics(recommended, relevant, k):
    items = list(recommended[:k])
    relevant = set(relevant)
    hits = np.array([item in relevant for item in items], dtype=float)
    hits = np.pad(hits, (0, max(0, k - len(hits))))[:k]
    discounts = np.log2(np.arange(2, k + 2))
    idcg = np.sum(np.ones(min(len(relevant), k)) / discounts[:min(len(relevant), k)])
    positions = np.flatnonzero(hits)
    return {
        'PrecisionAtK': hits.sum() / k,
        'RecallAtK': hits.sum() / len(relevant),
        'NDCGAtK': np.sum(hits / discounts) / idcg if idcg else 0.0,
        'MAPAtK': sum(hits[:p + 1].sum() / (p + 1) for p in positions) / min(len(relevant), k),
        'HitRateAtK': float(hits.sum() > 0),
    }

def evaluate(recommendations, query_ids, relevance_column, candidate_policy, ks=METRIC_KS):
    relevance_mask = (
        future_problem['learner_id'].isin(set(query_ids))
        & future_problem['in_candidate_catalog']
        & future_problem[relevance_column].eq(1)
    )
    if candidate_policy == 'novel_only':
        relevance_mask &= ~future_problem['seen_in_early'].fillna(False).astype(bool)
    relevant = future_problem.loc[relevance_mask, ['learner_id', 'item_id']].drop_duplicates()
    relevant_by_user = relevant.groupby('learner_id')['item_id'].agg(set).to_dict()
    eligible = set(query_ids) & set(relevant_by_user)
    ranked = recommendations.groupby('learner_id')['item_id'].agg(list).to_dict()
    rows = []
    for k in ks:
        values = []
        union = set()
        lengths = []
        for learner_id in sorted(eligible):
            items=ranked.get(learner_id,[])[:k]
            values.append(user_metrics(items,relevant_by_user[learner_id],k))
            union.update(items)
            lengths.append(len(items))
        means=pd.DataFrame(values).mean().to_dict()
        rows.append({**means,'K':k,'CatalogCoverageAtK':len(union)/len(catalog),
                     'MeanRecommendations': float(np.mean(lengths)),
                     'EvaluatedLearners': len(eligible)})
    return pd.DataFrame(rows)


## Step 6 — Discovery-only gate and reranker selection

In [21]:
BASE_SEQUENCE_WEIGHTS = (0.0, 0.10, 0.25, 0.40, 0.60)
MINIMUM_TRANSITION_SUPPORTS = (1, 5, 10, 20)
CONTENT_BONUSES = (0.0, 0.02, 0.05, 0.10)
configurations = []
for sequence_weight, minimum_support, content_bonus in product(
    BASE_SEQUENCE_WEIGHTS, MINIMUM_TRANSITION_SUPPORTS, CONTENT_BONUSES
):
    if sequence_weight == 0.0 and minimum_support != MINIMUM_TRANSITION_SUPPORTS[0]:
        continue
    configurations.append((sequence_weight, minimum_support, content_bonus))

tuning_rows = []
for sequence_weight, minimum_support, content_bonus in configurations:
    recommendations = gated_fusion(
        tuning_cf, tuning_sequence, tuning_content, discovery_tuning_ids,
        'all_supported', sequence_weight, minimum_support, content_bonus,
    )
    metrics = evaluate(
        recommendations, discovery_tuning_ids, 'relevance_binary',
        'all_supported', ks=(PRIMARY_K,),
    ).iloc[0]
    top10 = recommendations[recommendations['rank'].le(10)]
    tuning_rows.append({
        'BaseSequenceWeight': sequence_weight,
        'MinimumTransitionSupport': minimum_support,
        'ContentBonus': content_bonus,
        'ParameterSignature': (
            f'seq={sequence_weight:.2f}|support={minimum_support:02d}|content={content_bonus:.2f}'
        ),
        **metrics.to_dict(),
        'CFContributionRowsAt10': int(top10['cf_contribution'].gt(0).sum()),
        'SequentialContributionRowsAt10': int(top10['sequential_contribution'].gt(0).sum()),
        'ContentContributionRowsAt10': int(top10['content_contribution'].gt(0).sum()),
        'LearnersWithCFContributionAt10': int(top10.loc[
            top10['cf_contribution'].gt(0), 'learner_id'
        ].nunique()),
        'LearnersWithSequentialContributionAt10': int(top10.loc[
            top10['sequential_contribution'].gt(0), 'learner_id'
        ].nunique()),
        'LearnersWithJointCFSequentialAt10': int(top10.loc[
            top10['cf_contribution'].gt(0) & top10['sequential_contribution'].gt(0),
            'learner_id',
        ].nunique()),
        'LearnersWithContentContributionAt10': int(top10.loc[
            top10['content_contribution'].gt(0), 'learner_id'
        ].nunique()),
        'FallbackLearners': int(recommendations.loc[
            recommendations['FallbackUsed'], 'learner_id'
        ].nunique()),
    })
gated_tuning_metrics = pd.DataFrame(tuning_rows)
gated_tuning_metrics.to_csv(
    GATED_OUTPUT_ROOT / 'gated_tuning_metrics.csv', index=False
)

selection_rules = {
    'gated_unconstrained': pd.Series(True, index=gated_tuning_metrics.index),
    'gated_required_cf_sequential': (
        gated_tuning_metrics['BaseSequenceWeight'].gt(0)
        & gated_tuning_metrics['ContentBonus'].eq(0)
        & gated_tuning_metrics['LearnersWithCFContributionAt10'].gt(0)
        & gated_tuning_metrics['LearnersWithSequentialContributionAt10'].gt(0)
        & gated_tuning_metrics['LearnersWithJointCFSequentialAt10'].gt(0)
    ),
    'gated_required_cf_sequential_content': (
        gated_tuning_metrics['BaseSequenceWeight'].gt(0)
        & gated_tuning_metrics['ContentBonus'].gt(0)
        & gated_tuning_metrics['LearnersWithCFContributionAt10'].gt(0)
        & gated_tuning_metrics['LearnersWithSequentialContributionAt10'].gt(0)
        & gated_tuning_metrics['LearnersWithJointCFSequentialAt10'].gt(0)
        & gated_tuning_metrics['LearnersWithContentContributionAt10'].gt(0)
    ),
}
selected_rows = []
for model, mask in selection_rules.items():
    candidates = gated_tuning_metrics[mask].sort_values(
        ['NDCGAtK', 'RecallAtK', 'CatalogCoverageAtK', 'ParameterSignature'],
        ascending=[False, False, False, True], kind='mergesort',
    )
    assert len(candidates), model
    selected = candidates.iloc[0].to_dict()
    selected['Model'] = model
    selected['Requirement'] = {
        'gated_unconstrained': 'accuracy_only',
        'gated_required_cf_sequential': 'nonzero_cf_and_sequential_capacity',
        'gated_required_cf_sequential_content': 'nonzero_cf_sequential_and_content_capacity',
    }[model]
    selected_rows.append(selected)
gated_selection = pd.DataFrame(selected_rows)
gated_selection.to_csv(
    GATED_OUTPUT_ROOT / 'gated_parameter_selection.csv', index=False
)
assert gated_selection.loc[
    gated_selection['Model'].ne('gated_unconstrained'), 'BaseSequenceWeight'
].gt(0).all()
assert gated_selection.loc[
    gated_selection['Model'].eq('gated_required_cf_sequential_content'), 'ContentBonus'
].gt(0).all()
display(gated_selection[[
    'Model', 'ParameterSignature', 'NDCGAtK', 'RecallAtK',
    'CatalogCoverageAtK', 'LearnersWithCFContributionAt10',
    'LearnersWithSequentialContributionAt10', 'LearnersWithJointCFSequentialAt10',
    'LearnersWithContentContributionAt10', 'FallbackLearners',
]])
print(f'Tested {len(gated_tuning_metrics)} confidence-gated configurations.')

,Model,ParameterSignature,NDCGAtK,RecallAtK,CatalogCoverageAtK,LearnersWithCFContributionAt10,LearnersWithSequentialContributionAt10,LearnersWithJointCFSequentialAt10,LearnersWithContentContributionAt10,FallbackLearners
0,gated_unconstrained,seq=0.00|support=01|content=0.02,0.499094,0.266164,0.197365,5145,58,0,524,114
1,gated_required_cf_sequential,seq=0.10|support=01|content=0.00,0.494785,0.264936,0.198296,5145,3025,2910,0,131
2,gated_required_cf_sequential_content,seq=0.10|support=01|content=0.02,0.494981,0.264964,0.200282,5145,3028,2914,531,114


Tested 68 confidence-gated configurations.


## Step 7 — Refit components on all discovery learners and generate frozen validation variants

In [22]:
validation_sequence = sequential_candidates(
    validation_ids, BASELINE_ROOT / 'step11_validation_recent_items.parquet',
)
validation_content = content_candidates(validation_ids)
validation_cf = {
    'attempted': neighbor_candidates(discovery_ids, validation_ids, 'relevance_binary'),
    'successful': neighbor_candidates(discovery_ids, validation_ids, 'successful_future_item'),
}

recommendation_parts = []
for selected in gated_selection.itertuples(index=False):
    for relevance_name, relevance_column in {
        'attempted': 'relevance_binary', 'successful': 'successful_future_item'
    }.items():
        for candidate_policy in ['all_supported', 'novel_only']:
            recommendations = gated_fusion(
                validation_cf[relevance_name], validation_sequence, validation_content,
                validation_ids, candidate_policy, float(selected.BaseSequenceWeight),
                int(selected.MinimumTransitionSupport), float(selected.ContentBonus),
            )
            recommendations.insert(0, 'RelevanceDefinition', relevance_name)
            recommendations.insert(0, 'CandidatePolicy', candidate_policy)
            recommendations.insert(0, 'Model', selected.Model)
            recommendation_parts.append(recommendations)
gated_recommendations = pd.concat(recommendation_parts, ignore_index=True)
gated_recommendations.to_parquet(
    GATED_OUTPUT_ROOT / 'gated_validation_recommendations.parquet',
    index=False, compression='snappy',
)
assert not gated_recommendations.duplicated([
    'Model', 'CandidatePolicy', 'RelevanceDefinition', 'learner_id', 'item_id'
]).any()
assert set(gated_recommendations['learner_id']).issubset(set(validation_ids))
display(gated_recommendations.groupby([
    'Model', 'CandidatePolicy', 'RelevanceDefinition'
]).agg(Rows=('item_id', 'size'), Recipients=('learner_id', 'nunique'),
       FallbackRows=('FallbackUsed', 'sum')).reset_index())

,Model,CandidatePolicy,RelevanceDefinition,Rows,Recipients,FallbackRows
0,gated_required_cf_sequential,all_supported,attempted,126107,6667,3280
1,gated_required_cf_sequential,all_supported,successful,125509,6667,3280
2,gated_required_cf_sequential,novel_only,attempted,124388,6667,6940
3,gated_required_cf_sequential,novel_only,successful,123855,6667,7080
4,gated_required_cf_sequential_content,all_supported,attempted,127278,6667,2820
5,gated_required_cf_sequential_content,all_supported,successful,126795,6667,2820
6,gated_required_cf_sequential_content,novel_only,attempted,125988,6667,6200
7,gated_required_cf_sequential_content,novel_only,successful,125641,6667,6320
8,gated_unconstrained,all_supported,attempted,125884,6667,2820
9,gated_unconstrained,all_supported,successful,125070,6667,2820


## Step 8 — Final ranking, cold-start, contribution and baseline comparison

In [23]:
def evaluate_segments(recommendations, relevance_column, candidate_policy):
    relevance_mask = future_problem['in_candidate_catalog'] & future_problem[relevance_column].eq(1)
    if candidate_policy == 'novel_only':
        relevance_mask &= ~future_problem['seen_in_early'].fillna(False).astype(bool)
    relevant = future_problem.loc[relevance_mask, ['learner_id', 'item_id']].drop_duplicates()
    relevant_by_user = relevant.groupby('learner_id')['item_id'].agg(set).to_dict()
    validation = learners[learners['cohort'].eq('validation')].copy()
    eligible = (
        set(validation.loc[validation['evaluable_problem'], 'learner_id'])
        & set(relevant_by_user)
    )
    segments = {
        'all': eligible,
        'cold_start': eligible & set(validation.loc[
            validation['cold_start_problem_history'], 'learner_id'
        ]),
        'non_cold_start': eligible & set(validation.loc[
            ~validation['cold_start_problem_history'], 'learner_id'
        ]),
    }
    ranked = recommendations.sort_values(['learner_id', 'rank', 'item_id'], kind='mergesort')
    ranked_by_user = ranked.groupby('learner_id')['item_id'].agg(list).to_dict()
    rows=[]
    for segment, users in segments.items():
        if not users: continue
        for k in METRIC_KS:
            values = []
            union = set()
            lengths = []
            for learner_id in sorted(users):
                items=ranked_by_user.get(learner_id,[])[:k]
                values.append(user_metrics(items,relevant_by_user[learner_id],k))
                union.update(items)
                lengths.append(len(items))
            means=pd.DataFrame(values).mean().to_dict()
            rows.append({**means,'Segment':segment,'K':k,
                         'CatalogCoverageAtK':len(union)/len(catalog),
                         'MeanRecommendations':float(np.mean(lengths)),
                         'EvaluatedLearners':len(users)})
    return pd.DataFrame(rows)

metric_parts = []
diagnostic_rows = []
for (model, policy, relevance_name), frame in gated_recommendations.groupby([
    'Model', 'CandidatePolicy', 'RelevanceDefinition'
]):
    relevance_column = (
        'relevance_binary'
        if relevance_name == 'attempted'
        else 'successful_future_item'
    )
    metrics=evaluate_segments(frame,relevance_column,policy)
    metrics.insert(0,'RelevanceDefinition',relevance_name)
    metrics.insert(0,'CandidatePolicy',policy)
    metrics.insert(0,'Model',model)
    metric_parts.append(metrics)
    per_user=frame.groupby('learner_id').agg(
        RecommendationCount=('item_id','size'),
        FallbackUsed=('FallbackUsed','first'),
        CFContribution=('cf_contribution','sum'),
        SequentialContribution=('sequential_contribution','sum'),
        ContentContribution=('content_contribution','sum'),
    ).reset_index()
    audit=learners[learners['cohort'].eq('validation')][[
        'learner_id','cold_start_problem_history'
    ]].merge(per_user,on='learner_id',how='left')
    audit['RecommendationCount']=audit['RecommendationCount'].fillna(0).astype(int)
    audit['FallbackUsed']=audit['FallbackUsed'].fillna(False).astype(bool)
    audit[['CFContribution','SequentialContribution','ContentContribution']]=audit[[
        'CFContribution','SequentialContribution','ContentContribution'
    ]].fillna(0.0)
    for segment,mask in {
        'all':pd.Series(True,index=audit.index),
        'cold_start':audit['cold_start_problem_history'],
        'non_cold_start':~audit['cold_start_problem_history'],
    }.items():
        subset=audit[mask]
        diagnostic_rows.append({
            'Model':model,'CandidatePolicy':policy,'RelevanceDefinition':relevance_name,
            'Segment':segment,'Learners':len(subset),
            'FallbackLearners':subset['FallbackUsed'].sum(),
            'NoRecommendationLearners':subset['RecommendationCount'].eq(0).sum(),
            'LearnersUsingCF':subset['CFContribution'].gt(0).sum(),
            'LearnersUsingSequential':subset['SequentialContribution'].gt(0).sum(),
            'LearnersUsingBothCFSequential':(
                subset['CFContribution'].gt(0) & subset['SequentialContribution'].gt(0)
            ).sum(),
            'LearnersUsingContent':subset['ContentContribution'].gt(0).sum(),
            'MeanRecommendations':subset['RecommendationCount'].mean(),
        })
gated_metrics=pd.concat(metric_parts,ignore_index=True)
gated_diagnostics=pd.DataFrame(diagnostic_rows)
gated_metrics.to_csv(
    GATED_OUTPUT_ROOT / 'gated_metrics.csv', index=False
)
gated_diagnostics.to_csv(
    GATED_OUTPUT_ROOT / 'gated_cold_start_and_contribution_diagnostics.csv',
    index=False,
)

comparison=gated_metrics[
    gated_metrics['CandidatePolicy'].eq('all_supported')
    & gated_metrics['RelevanceDefinition'].eq('attempted')
    & gated_metrics['Segment'].eq('all')
    & gated_metrics['K'].eq(10)
].copy()
phase3_metrics=pd.read_csv(BASELINE_ROOT/'recommendation_metrics.csv')
cf=phase3_metrics[
    phase3_metrics['Dataset'].eq('ASSISTments')
    & phase3_metrics['Task'].eq('problem')
    & phase3_metrics['Model'].eq('learner_neighbor_cf')
    & phase3_metrics['CandidatePolicy'].eq('all_supported')
    & phase3_metrics['RelevanceDefinition'].eq('attempted')
    & phase3_metrics['Segment'].eq('all')
    & phase3_metrics['K'].eq(10)
].iloc[0]
global_metrics=pd.read_csv(GLOBAL_HYBRID_ROOT/'hybrid_metrics.csv')
global_row=global_metrics[
    global_metrics['Dataset'].eq('ASSISTments')
    & global_metrics['Task'].eq('problem')
    & global_metrics['Model'].eq('hybrid_neighbor_sequential')
    & global_metrics['CandidatePolicy'].eq('all_supported')
    & global_metrics['RelevanceDefinition'].eq('attempted')
    & global_metrics['Segment'].eq('all')
    & global_metrics['K'].eq(10)
].iloc[0]
comparison['RecallDeltaVsCF']=comparison['RecallAtK']-cf['RecallAtK']
comparison['NDCGDeltaVsCF']=comparison['NDCGAtK']-cf['NDCGAtK']
comparison['CoverageRatioVsCF']=comparison['CatalogCoverageAtK']/cf['CatalogCoverageAtK']
comparison['RecallDeltaVsGlobalHybrid']=comparison['RecallAtK']-global_row['RecallAtK']
comparison['NDCGDeltaVsGlobalHybrid']=comparison['NDCGAtK']-global_row['NDCGAtK']
comparison['PointGateVsCF']=(
    comparison['RecallDeltaVsCF'].gt(0)
    & comparison['NDCGDeltaVsCF'].gt(0)
    & comparison['CoverageRatioVsCF'].ge(COVERAGE_FLOOR_RATIO)
)
comparison.to_csv(
    GATED_OUTPUT_ROOT / 'gated_model_comparison.csv', index=False
)
display(comparison[[
    'Model','RecallAtK','NDCGAtK','CatalogCoverageAtK',
    'RecallDeltaVsCF','NDCGDeltaVsCF','CoverageRatioVsCF',
    'RecallDeltaVsGlobalHybrid','NDCGDeltaVsGlobalHybrid','PointGateVsCF'
]])
display(gated_diagnostics[
    gated_diagnostics['CandidatePolicy'].eq('all_supported')
    & gated_diagnostics['RelevanceDefinition'].eq('attempted')
].sort_values(['Model','Segment']))

,Model,RecallAtK,NDCGAtK,CatalogCoverageAtK,RecallDeltaVsCF,NDCGDeltaVsCF,CoverageRatioVsCF,RecallDeltaVsGlobalHybrid,NDCGDeltaVsGlobalHybrid,PointGateVsCF
1,gated_required_cf_sequential,0.268331,0.494173,0.223585,-0.001806,-0.004219,1.018552,-0.001806,-0.004219,False
37,gated_required_cf_sequential_content,0.268461,0.494259,0.226627,-0.001675,-0.004132,1.032410,-0.001675,-0.004132,False
73,gated_unconstrained,0.270500,0.498745,0.224088,0.000363,0.000353,1.020843,0.000363,0.000353,True


,Model,CandidatePolicy,RelevanceDefinition,Segment,Learners,FallbackLearners,NoRecommendationLearners,LearnersUsingCF,LearnersUsingSequential,LearnersUsingBothCFSequential,LearnersUsingContent,MeanRecommendations
0,gated_required_cf_sequential,all_supported,attempted,all,6667,164,0,6377,4432,4306,0,18.915104
1,gated_required_cf_sequential,all_supported,attempted,cold_start,164,164,0,0,0,0,0,20.000000
2,gated_required_cf_sequential,all_supported,attempted,non_cold_start,6503,0,0,6377,4432,4306,0,18.887744
12,gated_required_cf_sequential_content,all_supported,attempted,all,6667,141,0,6377,4431,4305,1018,19.090745
13,gated_required_cf_sequential_content,all_supported,attempted,cold_start,164,141,0,0,0,0,23,20.000000
14,gated_required_cf_sequential_content,all_supported,attempted,non_cold_start,6503,0,0,6377,4431,4305,995,19.067815
24,gated_unconstrained,all_supported,attempted,all,6667,141,0,6377,126,0,1015,18.881656
25,gated_unconstrained,all_supported,attempted,cold_start,164,141,0,0,0,0,23,20.000000
26,gated_unconstrained,all_supported,attempted,non_cold_start,6503,0,0,6377,126,0,992,18.853452


## Step 9 — Save the gated experiment contract and artifact manifest

In [24]:
gated_config = {
    'experiment':'confidence_gated_hybrid','dataset':'ASSISTments','task':'problem',
    'candidate_depth':CANDIDATE_DEPTH,'final_k':FINAL_K,'rrf_constant':RRF_CONSTANT,
    'selection_source':'discovery_only','selection_metric':'attempted_all_supported_NDCGAt10',
    'variants':list(gated_selection['Model']),
    'required_hybrid_variants_enforce_nonzero_sequence_capacity':True,
    'content_role':'bounded_reranker','fallback':'discovery_early_popularity',
    'validation_scope':'frozen_cohort_post_selection_not_independent_test',
    'novel_relevance_excludes_seen':True,'dense_learner_item_matrix_constructed':False,
}
with open(
    GATED_OUTPUT_ROOT / 'gated_experiment_config.json',
    'w', encoding='utf-8',
) as file:
    json.dump(gated_config, file, indent=2)
gated_paths = {
    'split': GATED_OUTPUT_ROOT / 'gated_experiment_split.csv',
    'tuning_components': GATED_TUNING_COMPONENT_PATH,
    'complementarity': GATED_OUTPUT_ROOT / 'component_complementarity.csv',
    'tuning_metrics': GATED_OUTPUT_ROOT / 'gated_tuning_metrics.csv',
    'selection': GATED_OUTPUT_ROOT / 'gated_parameter_selection.csv',
    'recommendations': (
        GATED_OUTPUT_ROOT / 'gated_validation_recommendations.parquet'
    ),
    'metrics': GATED_OUTPUT_ROOT / 'gated_metrics.csv',
    'diagnostics': (
        GATED_OUTPUT_ROOT
        / 'gated_cold_start_and_contribution_diagnostics.csv'
    ),
    'comparison': GATED_OUTPUT_ROOT / 'gated_model_comparison.csv',
    'config': GATED_OUTPUT_ROOT / 'gated_experiment_config.json',
}
gated_manifest = pd.DataFrame([{
    'Artifact':name,'File':path.name,
    'Rows':pq.ParquetFile(path).metadata.num_rows if path.suffix=='.parquet'
           else (len(pd.read_csv(path)) if path.suffix=='.csv' else 1),
    'Bytes':path.stat().st_size,
} for name, path in gated_paths.items()])
gated_manifest.to_csv(
    GATED_OUTPUT_ROOT / 'artifact_manifest.csv', index=False
)
assert len(gated_manifest) == len(gated_paths)
display(gated_manifest)

,Artifact,File,Rows,Bytes
0,split,gated_experiment_split.csv,33335,802382
1,tuning_components,gated_tuning_components.parquet,963473,6649393
2,complementarity,component_complementarity.csv,4799,123547
3,tuning_metrics,gated_tuning_metrics.csv,68,15935
4,selection,gated_parameter_selection.csv,3,1296
5,recommendations,gated_validation_recommendations.parquet,1506099,17443605
6,metrics,gated_metrics.csv,108,20858
7,diagnostics,gated_cold_start_and_contribution_diagnostics.csv,36,3823
8,comparison,gated_model_comparison.csv,3,1195
9,config,gated_experiment_config.json,1,704


# Part C — Final promotion, uncertainty, audit and artifact contract

The discovery-selected unconstrained gated router becomes the project architecture. A paired 2,000-resample learner bootstrap distinguishes that architectural choice from a statistically reliable improvement claim. The global cluster variants remain historical ablations; no separate content-increment experiment is added.

In [25]:
FINAL_OUTPUT_ROOT = Path(
    '/content/drive/MyDrive/datasets/hybrid_recommender_gated_final_outputs'
)
FINAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PROMOTED = 'gated_unconstrained'
PRODUCTION_NAME = 'gated_cf_content_with_sequential_fallback'
selected = gated_selection[gated_selection['Model'].eq(PROMOTED)].iloc[0]
assert np.isclose(selected['BaseSequenceWeight'],0.0)
assert int(selected['MinimumTransitionSupport'])==1
assert np.isclose(selected['ContentBonus'],0.02)
promoted = gated_recommendations[
    gated_recommendations['Model'].eq(PROMOTED)
].copy()
primary = promoted[
    promoted['CandidatePolicy'].eq('all_supported')
    & promoted['RelevanceDefinition'].eq('attempted')
].copy()
relevant = future_problem[
    future_problem['in_candidate_catalog']
    & future_problem['relevance_binary'].eq(1)
][['learner_id', 'item_id']].drop_duplicates()
relevant_by_user = relevant.groupby('learner_id')['item_id'].agg(set).to_dict()
eligible = sorted(
    set(learners.loc[
        learners['cohort'].eq('validation') & learners['evaluable_problem'],
        'learner_id',
    ]) & set(relevant_by_user)
)

def per_user_table(frame, model):
    ranked = (
        frame.sort_values(['learner_id', 'rank', 'item_id'], kind='mergesort')
        .groupby('learner_id')['item_id'].agg(list).to_dict()
    )
    rows = []
    for learner_id in eligible:
        values = user_metrics(
            ranked.get(learner_id, []), relevant_by_user[learner_id], 10
        )
        rows.append({
            'learner_id': learner_id, 'Model': model,
            'RecallAt10': values['RecallAtK'],
            'NDCGAt10': values['NDCGAtK'],
        })
    return pd.DataFrame(rows)
phase3_ds=pds.dataset(BASELINE_ROOT/'validation_recommendations.parquet',format='parquet')
cf_filter = (
    (pds.field('Dataset') == 'ASSISTments')
    & (pds.field('Task') == 'problem')
    & (pds.field('Model') == 'learner_neighbor_cf')
    & (pds.field('CandidatePolicy') == 'all_supported')
    & (pds.field('RelevanceDefinition') == 'attempted')
)
cf_rows=phase3_ds.to_table(filter=cf_filter,columns=['learner_id','item_id','rank']).to_pandas()
paired = pd.concat([
    per_user_table(primary, PRODUCTION_NAME),
    per_user_table(cf_rows, 'learner_neighbor_cf'),
], ignore_index=True)
paired.to_csv(FINAL_OUTPUT_ROOT/'paired_user_metrics.csv',index=False)
left=paired[paired['Model'].eq(PRODUCTION_NAME)].set_index('learner_id')
right=paired[paired['Model'].eq('learner_neighbor_cf')].set_index('learner_id').loc[left.index]
rng = np.random.default_rng(RANDOM_STATE)
boot = []
for metric in ['RecallAt10','NDCGAt10']:
    delta = (left[metric] - right[metric]).to_numpy()
    means = np.empty(2000)
    for sample_index in range(2000):
        indices = rng.integers(0, len(delta), len(delta))
        means[sample_index] = delta[indices].mean()
    lower, upper = np.quantile(means, [0.025, 0.975])
    boot.append({
        'Metric': metric, 'MeanPairedDifference': delta.mean(),
        'BootstrapLower95': lower, 'BootstrapUpper95': upper,
        'SamplesAboveZero': (means > 0).mean(),
        'EvaluatedLearners': len(delta), 'Resamples': 2000,
        'Seed': RANDOM_STATE,
    })
bootstrap = pd.DataFrame(boot)
bootstrap.to_csv(
    FINAL_OUTPUT_ROOT / 'hybrid_bootstrap_differences.csv', index=False
)
point=comparison[comparison['Model'].eq(PROMOTED)].iloc[0]
ndcg_lower=bootstrap.set_index('Metric').loc['NDCGAt10','BootstrapLower95']
reliable = bool(
    point['RecallDeltaVsCF'] > 0
    and point['NDCGDeltaVsCF'] > 0
    and ndcg_lower > 0
    and point['CoverageRatioVsCF'] >= COVERAGE_FLOOR_RATIO
)
decision = pd.DataFrame([{
    'ProductionModel': PRODUCTION_NAME,
    'ProjectArchitectureSelected': True,
    'StatisticallyReliableImprovementVsCF': reliable,
    'RecallDeltaVsCF': point['RecallDeltaVsCF'],
    'NDCGDeltaVsCF': point['NDCGDeltaVsCF'],
    'NDCGBootstrapLower95': ndcg_lower,
    'Interpretation': (
        'selected_with_reliable_improvement' if reliable
        else 'selected_without_reliable_improvement_claim'
    ),
}])
decision.to_csv(FINAL_OUTPUT_ROOT/'production_decision.csv',index=False)
display(bootstrap)
display(decision)


,Metric,MeanPairedDifference,BootstrapLower95,BootstrapUpper95,SamplesAboveZero,EvaluatedLearners,Resamples,Seed
0,RecallAt10,0.000363,-0.000008,0.000871,0.9625,5996,2000,42
1,NDCGAt10,0.000353,-0.000034,0.000905,0.9430,5996,2000,42


,ProductionModel,ProjectArchitectureSelected,StatisticallyReliableImprovementVsCF,RecallDeltaVsCF,NDCGDeltaVsCF,NDCGBootstrapLower95,Interpretation
0,gated_cf_content_with_sequential_fallback,True,False,0.000363,0.000353,-0.000034,selected_without_reliable_improvement_claim


## Step 12 — Preserve cluster evidence as a historical ablation

The global-fusion cluster variants remain excluded and cannot claim a gated-architecture cluster increment.

In [26]:
cluster_ablation = hybrid_point_comparison[
    hybrid_point_comparison['Dataset'].eq('ASSISTments')
    & hybrid_point_comparison['Task'].eq('problem')
    & hybrid_point_comparison['Model'].isin([
        'hybrid_with_same_cluster_neighbor',
        'hybrid_with_cluster_popularity',
    ])
].copy()
cluster_ablation['Decision'] = 'exclude_cluster_signal'
cluster_ablation['Scope'] = 'historical_global_fusion_ablation'
cluster_ablation.to_csv(FINAL_OUTPUT_ROOT/'cluster_ablation_decisions.csv',index=False)
display(cluster_ablation[['Model', 'Decision', 'Scope']])


,Model,Decision,Scope
2,hybrid_with_same_cluster_neighbor,exclude_cluster_signal,historical_global_fusion_ablation
3,hybrid_with_cluster_popularity,exclude_cluster_signal,historical_global_fusion_ablation


## Step 13 — Audit leakage, routing and provenance

These assertions validate the promoted router, including novel-only exclusion and sequence use only when CF is unavailable.

In [27]:
seen = set(map(tuple, early_problem[
    early_problem['learner_id'].isin(validation_ids)
    & early_problem['in_candidate_catalog']
][['learner_id', 'item_id']].drop_duplicates().to_numpy()))
novel = promoted[promoted['CandidatePolicy'].eq('novel_only')]
fallback = promoted[promoted['FallbackUsed']]
audit = {
    'DiscoveryValidationDisjoint': set(discovery_ids).isdisjoint(validation_ids),
    'RecommendationsValidationOnly': set(promoted['learner_id']).issubset(validation_ids),
    'CandidatesInFrozenCatalog': set(promoted['item_id']).issubset(item_order),
    'NovelRecommendationsExcludeSeen': not any(
        pair in seen for pair in map(
            tuple, novel[['learner_id', 'item_id']].to_numpy()
        )
    ),
    'UniqueRows': not promoted.duplicated([
        'CandidatePolicy', 'RelevanceDefinition', 'learner_id', 'item_id'
    ]).any(),
    'FiniteScores': np.isfinite(promoted['score']).all(),
    'RanksWithin20': promoted['rank'].between(1, 20).all(),
    'FallbackContributionsZero': fallback[[
        'cf_contribution', 'sequential_contribution', 'content_contribution'
    ]].fillna(0).eq(0).all().all(),
    'NoJointCFSequential': not (
        promoted['cf_contribution'].gt(0)
        & promoted['sequential_contribution'].gt(0)
    ).any(),
    'SequenceOnlyWithoutCF': not promoted.loc[
        promoted['sequential_contribution'].gt(0), 'cf_contribution'
    ].gt(0).any(),
    'NovelRelevanceExcludesSeen': True,
    'DenseMatrixConstructed': False,
}
assert all(
    value for key, value in audit.items()
    if key != 'DenseMatrixConstructed'
)
assert audit['DenseMatrixConstructed'] is False
audit_frame = pd.DataFrame([{
    **audit, 'RecommendationRowsChecked': len(promoted)
}])
audit_frame.to_csv(FINAL_OUTPUT_ROOT / 'leakage_audit.csv', index=False)
display(audit_frame.T)


,0
DiscoveryValidationDisjoint,True
RecommendationsValidationOnly,True
CandidatesInFrozenCatalog,True
NovelRecommendationsExcludeSeen,True
UniqueRows,True
FiniteScores,True
RanksWithin20,True
FallbackContributionsZero,True
NoJointCFSequential,True
SequenceOnlyWithoutCF,True


## Step 14 — Write the canonical Phase 4 artifact contract

This cell writes only the promoted architecture as the final recommendation payload and verifies the manifest.

In [28]:
promoted.insert(0, 'ProductionModel', PRODUCTION_NAME)
promoted.to_parquet(
    FINAL_OUTPUT_ROOT / 'hybrid_recommendations.parquet',
    index=False, compression='snappy',
)
contribution_columns = [
    'ProductionModel', 'CandidatePolicy', 'RelevanceDefinition',
    'learner_id', 'item_id', 'rank', 'score', 'score_source',
    'FallbackUsed', 'cf_contribution', 'sequential_contribution',
    'content_contribution', 'effective_cf_weight',
    'effective_sequence_weight',
]
promoted[contribution_columns].to_parquet(
    FINAL_OUTPUT_ROOT / 'hybrid_component_contributions.parquet',
    index=False, compression='snappy',
)
final_metrics = gated_metrics[gated_metrics['Model'].eq(PROMOTED)].copy()
final_metrics.to_csv(FINAL_OUTPUT_ROOT / 'hybrid_metrics.csv', index=False)
coverage_columns = [
    'Model', 'CandidatePolicy', 'RelevanceDefinition', 'Segment', 'K',
    'CatalogCoverageAtK', 'MeanRecommendations', 'EvaluatedLearners',
]
final_metrics[coverage_columns].to_csv(
    FINAL_OUTPUT_ROOT / 'hybrid_coverage.csv', index=False
)
gated_diagnostics[gated_diagnostics['Model'].eq(PROMOTED)].to_csv(
    FINAL_OUTPUT_ROOT / 'hybrid_cold_start_diagnostics.csv', index=False
)
gated_tuning_metrics.to_csv(
    FINAL_OUTPUT_ROOT / 'hybrid_tuning_metrics.csv', index=False
)
gated_selection.to_csv(
    FINAL_OUTPUT_ROOT / 'hybrid_parameter_selection.csv', index=False
)
comparison[comparison['Model'].eq(PROMOTED)].to_csv(
    FINAL_OUTPUT_ROOT / 'hybrid_model_comparison.csv', index=False
)
complementarity.to_csv(
    FINAL_OUTPUT_ROOT / 'component_complementarity.csv', index=False
)
config = {
    'phase': 4, 'production_model': PRODUCTION_NAME,
    'source_variant': PROMOTED, 'project_architecture_selected': True,
    'selection_source': 'discovery_only', 'base_sequence_weight': 0.0,
    'minimum_transition_support': 1, 'content_bonus': 0.02,
    'candidate_depth': 100, 'final_k': 20,
    'collaborative_role': 'primary_ranker',
    'content_role': 'bounded_reranker',
    'sequence_role': 'route_when_cf_unavailable',
    'popularity_role': 'final_fallback',
    'global_fusion_role': 'historical_comparator_and_cluster_ablation',
    'statistically_reliable_improvement_vs_cf': reliable,
    'validation_scope': 'frozen_cohort_post_selection_not_independent_test',
    'novel_relevance_excludes_seen': True,
    'dense_learner_item_matrix_constructed': False,
    'content_increment_experiment_included': False,
}
with open(
    FINAL_OUTPUT_ROOT / 'phase4_config.json', 'w', encoding='utf-8'
) as file:
    json.dump(config, file, indent=2)
names = [
    'hybrid_tuning_metrics.csv', 'hybrid_parameter_selection.csv',
    'hybrid_metrics.csv', 'hybrid_coverage.csv',
    'hybrid_model_comparison.csv', 'hybrid_bootstrap_differences.csv',
    'production_decision.csv', 'hybrid_cold_start_diagnostics.csv',
    'hybrid_recommendations.parquet',
    'hybrid_component_contributions.parquet',
    'component_complementarity.csv', 'paired_user_metrics.csv',
    'cluster_ablation_decisions.csv', 'leakage_audit.csv',
    'phase4_config.json',
]
manifest_rows = []
for name in names:
    path = FINAL_OUTPUT_ROOT / name
    if path.suffix == '.parquet':
        rows = pq.ParquetFile(path).metadata.num_rows
    elif path.suffix == '.csv':
        rows = len(pd.read_csv(path))
    else:
        rows = 1
    manifest_rows.append({
        'File': name, 'Rows': rows, 'Bytes': path.stat().st_size
    })
final_manifest = pd.DataFrame(manifest_rows)
final_manifest.to_csv(FINAL_OUTPUT_ROOT / 'artifact_manifest.csv', index=False)

display(final_manifest)

,File,Rows,Bytes
0,hybrid_tuning_metrics.csv,68,15935
1,hybrid_parameter_selection.csv,3,1296
2,hybrid_metrics.csv,36,7083
3,hybrid_coverage.csv,36,3486
4,hybrid_model_comparison.csv,1,572
5,hybrid_bootstrap_differences.csv,2,306
6,production_decision.csv,1,311
7,hybrid_cold_start_diagnostics.csv,12,1294
8,hybrid_recommendations.parquet,500538,4297535
9,hybrid_component_contributions.parquet,500538,1498331
